In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/abinavprabhuv/dasonet-eod-20260914-after-seed1/DASO-Net-EOD-20260914-AFTER-SEED1-STATE.json
/kaggle/input/datasets/abinavprabhuv/dasonet-eod-20260914-after-seed1/DASO-Net-EOD-20260914-AFTER-SEED1-KAGGLE-UPLOAD/DASO-Net-EOD-20260914-AFTER-SEED1/EOD_MANIFEST.json
/kaggle/input/datasets/abinavprabhuv/dasonet-eod-20260914-after-seed1/DASO-Net-EOD-20260914-AFTER-SEED1-KAGGLE-UPLOAD/DASO-Net-EOD-20260914-AFTER-SEED1/RESTORE_AFTER_SEED1.py
/kaggle/input/datasets/abinavprabhuv/dasonet-eod-20260914-after-seed1/DASO-Net-EOD-20260914-AFTER-SEED1-KAGGLE-UPLOAD/DASO-Net-EOD-20260914-AFTER-SEED1/EOD_STATE.json
/kaggle/input/datasets/abinavprabhuv/dasonet-eod-20260914-after-seed1/DASO-Net-EOD-20260914-AFTER-SEED1-KAGGLE-UPLOAD/DASO-Net-EOD-20260914-AFTER-SEED1/workspace/phase5c_baseline_curve1000updates_v1.json
/kaggle/input/datasets/abinavprabhuv/dasonet-eod-20260914-after-seed1/DASO-Net-EOD-20260914-AFTER-SEED1-KAGGLE-UPLOAD/DASO-Net-EOD-20260914-AFTER-SEED1/workspace/DASO-Ne

In [2]:
# ============================================================
# DASO-Net — AFTER-SEED1 RESTORE + SEED-2 READINESS AUDIT v2
#
# PURPOSE
# -------
# Restore the verified end-of-day handoff from Kaggle's
# automatically extracted dataset layout into /kaggle/working.
#
# IMPORTANT
# ---------
# Kaggle unpacked:
#   - the sanitized outer ZIP
#   - the renamed inner recipe ZIP
#
# Therefore the original byte-identical recipe ZIP no longer
# exists inside /kaggle/input. We DO NOT try to reconstruct it,
# because a newly-created ZIP would not have the original SHA.
#
# Instead we:
#   1. verify all surviving critical files against EOD_STATE
#   2. verify both extracted recipe-freeze trees are identical
#   3. restore the working tree
#   4. verify Seed 1 terminal SHA
#   5. verify frozen Seed 2 identity
#   6. verify Seed 2 output does NOT exist
#   7. verify 2 x T4 runtime
#
# NO TRAINING.
# ============================================================

from __future__ import annotations

import hashlib
import json
import shutil
from pathlib import Path

import torch


# ============================================================
# 1. FIXED INPUT PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

EOD_DATASET_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-eod-20260914-after-seed1"
)

HANDOFF_ROOT = (
    EOD_DATASET_ROOT
    / "DASO-Net-EOD-20260914-AFTER-SEED1-KAGGLE-UPLOAD"
    / "DASO-Net-EOD-20260914-AFTER-SEED1"
)

SOURCE_WORKSPACE = (
    HANDOFF_ROOT
    / "workspace"
)

SOURCE_EOD_STATE = (
    HANDOFF_ROOT
    / "EOD_STATE.json"
)

SOURCE_EOD_MANIFEST = (
    HANDOFF_ROOT
    / "EOD_MANIFEST.json"
)

TOPLEVEL_STATE = (
    EOD_DATASET_ROOT
    / "DASO-Net-EOD-20260914-AFTER-SEED1-STATE.json"
)


# ============================================================
# 2. SOURCE COMPONENTS
# ============================================================

SRC_CODE = (
    SOURCE_WORKSPACE
    / "DASO-Net-Phase5B-KaggleTraining"
)

SRC_MULTISEED = (
    SOURCE_WORKSPACE
    / "DASO-Net-FinalMultiSeed-v1"
)

SRC_FREEZE = (
    SOURCE_WORKSPACE
    / "DASO-Net-Phase5C-FinalTrainingRecipe-FREEZE-v1"
)

# Kaggle extracted the renamed copy of the original inner ZIP.
SRC_FREEZE_ARCHIVE_COPY = (
    SOURCE_WORKSPACE
    / "DASO-Net-Phase5C-FinalTrainingRecipe-FREEZE-v1-ORIGINAL-ARCHIVE"
    / "DASO-Net-Phase5C-FinalTrainingRecipe-FREEZE-v1"
)

SRC_BASE_CONFIG = (
    SOURCE_WORKSPACE
    / "phase5c_baseline_curve1000updates_v1.json"
)

SRC_PREFLIGHT = (
    SOURCE_WORKSPACE
    / "DASO-Net-FinalMultiSeed-Preflight-v1.json"
)

SRC_SEED1_RUN = (
    SOURCE_WORKSPACE
    / "dasonet_phase5b"
    / "dasonet_final_seed01_s1782656963"
)

SRC_SEED1_QC = (
    SOURCE_WORKSPACE
    / "dasonet_final_qc"
    / "seed01_s1782656963_val1000_v1"
)

SRC_DURATION = (
    SOURCE_WORKSPACE
    / "dasonet_phase5c"
    / "duration_selection_val1000_v1"
)

SRC_CURVE_SUMMARY = (
    SOURCE_WORKSPACE
    / "dasonet_phase5c"
    / "baseline_curve1000updates_v1"
    / "baseline_learning_curve_3000_summary.json"
)


# ============================================================
# 3. CRITICAL SOURCE FILES
# ============================================================

SRC_SEED1_TERMINAL = (
    SRC_SEED1_RUN
    / "snapshots"
    / "epoch_011_opt_002750.pt"
)

SRC_SEED1_TRAINING_MANIFEST = (
    SRC_SEED1_RUN
    / "FINAL_SEED01_TRAINING_MANIFEST.json"
)

SRC_SEED1_COMPLETION_MANIFEST = (
    SRC_SEED1_RUN
    / "FINAL_SEED01_AMP_SKIP_COMPLETION.json"
)

SRC_SEED1_QC_SUMMARY = (
    SRC_SEED1_QC
    / "FINAL_SEED01_QC_SUMMARY.json"
)

SRC_MULTISEED_PLAN = (
    SRC_MULTISEED
    / "FINAL_MULTISEED_PLAN_v1.json"
)

SRC_SEED1_CONFIG = (
    SRC_MULTISEED
    / "configs"
    / "final_seed01_s1782656963.json"
)

SRC_SEED2_CONFIG = (
    SRC_MULTISEED
    / "configs"
    / "final_seed02_s901395089.json"
)

SRC_SEED3_CONFIG = (
    SRC_MULTISEED
    / "configs"
    / "final_seed03_s252916394.json"
)

SRC_DECODER = (
    SRC_CODE
    / "src"
    / "dasonet"
    / "model"
    / "decoding.py"
)

SRC_TRAINER = (
    SRC_CODE
    / "scripts"
    / "train_phase5b.py"
)

SRC_LAUNCHER = (
    SRC_CODE
    / "scripts"
    / "launch_phase5c_checkpointed_baseline.py"
)


# ============================================================
# 4. EXPECTED SCIENTIFIC IDENTITY
# ============================================================

EXPECTED_SEED1 = 1782656963
EXPECTED_SEED2 = 901395089
EXPECTED_SEED3 = 252916394

EXPECTED_TARGET_UPDATES = 2750

EXPECTED_SEED1_TERMINAL_SHA = (
    "e4c4b79ad4b8c6e18ba2b1e3d6cbe796"
    "c9cc301bd6fcc7c596fc290ad3a7e81d"
)

EXPECTED_DECODER_SHA = (
    "9799eb1be568e3d6e0ce82b7c99df7a0"
    "ed05d271cd44ff24b612526cc18537d1"
)

# Historical provenance only.
# Kaggle unpacked the original archive, so there is no longer
# a byte-identical ZIP inside the mounted input.
EXPECTED_ORIGINAL_RECIPE_ZIP_SHA = (
    "b102deb8ec9de4be3816e4450a4dc7b2"
    "f78f5439634cee02fe4656d1a268bf3a"
)


# ============================================================
# 5. DESTINATIONS
# ============================================================

WORKING = Path(
    "/kaggle/working"
)

DST_CODE = (
    WORKING
    / "DASO-Net-Phase5B-KaggleTraining"
)

DST_MULTISEED = (
    WORKING
    / "DASO-Net-FinalMultiSeed-v1"
)

DST_FREEZE = (
    WORKING
    / "DASO-Net-Phase5C-FinalTrainingRecipe-FREEZE-v1"
)

DST_BASE_CONFIG = (
    WORKING
    / "phase5c_baseline_curve1000updates_v1.json"
)

DST_PREFLIGHT = (
    WORKING
    / "DASO-Net-FinalMultiSeed-Preflight-v1.json"
)

DST_SEED1_RUN = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed01_s1782656963"
)

DST_SEED1_QC = (
    WORKING
    / "dasonet_final_qc"
    / "seed01_s1782656963_val1000_v1"
)

DST_DURATION = (
    WORKING
    / "dasonet_phase5c"
    / "duration_selection_val1000_v1"
)

DST_CURVE_SUMMARY = (
    WORKING
    / "dasonet_phase5c"
    / "baseline_curve1000updates_v1"
    / "baseline_learning_curve_3000_summary.json"
)

RESTORE_AUDIT_JSON = (
    WORKING
    / "DASO-Net-AFTER-SEED1-RESTORE-AUDIT-v2.json"
)


# ============================================================
# 6. HELPERS
# ============================================================

def require(
    path: Path,
    label: str,
) -> None:

    if not path.exists():

        raise RuntimeError(
            f"{label} missing:\n{path}"
        )


def sha256_file(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):
            h.update(
                block
            )

    return h.hexdigest()


def load_json(
    path: Path,
):

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def copy_file(
    src: Path,
    dst: Path,
) -> None:

    dst.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        src,
        dst,
    )


def replace_tree(
    src: Path,
    dst: Path,
) -> None:

    if dst.exists():

        shutil.rmtree(
            dst
        )

    dst.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copytree(
        src,
        dst,
    )


def tree_hash_map(
    root: Path,
) -> dict[str, str]:

    result = {}

    for path in sorted(
        p
        for p in root.rglob("*")
        if p.is_file()
    ):

        relative = str(
            path.relative_to(
                root
            )
        )

        result[
            relative
        ] = sha256_file(
            path
        )

    return result


# ============================================================
# 7. INPUT PRESENCE
# ============================================================

print(
    "============================================================"
)

print(
    "DASO-Net — AFTER-SEED1 RESTORE + SEED-2 READINESS v2"
)

print(
    "============================================================"
)


required_inputs = [
    (
        PROJECT_ROOT,
        "Frozen project dataset",
    ),
    (
        EOD_DATASET_ROOT,
        "EOD Kaggle dataset",
    ),
    (
        HANDOFF_ROOT,
        "Extracted EOD handoff root",
    ),
    (
        SOURCE_WORKSPACE,
        "Extracted EOD workspace",
    ),
    (
        SOURCE_EOD_STATE,
        "EOD state",
    ),
    (
        SOURCE_EOD_MANIFEST,
        "EOD manifest",
    ),
    (
        TOPLEVEL_STATE,
        "Top-level EOD state",
    ),
    (
        SRC_CODE,
        "Validated code tree",
    ),
    (
        SRC_MULTISEED,
        "Frozen multi-seed directory",
    ),
    (
        SRC_FREEZE,
        "Primary extracted freeze tree",
    ),
    (
        SRC_FREEZE_ARCHIVE_COPY,
        "Archive-derived freeze tree",
    ),
    (
        SRC_SEED1_TERMINAL,
        "Seed-1 terminal model",
    ),
    (
        SRC_SEED1_TRAINING_MANIFEST,
        "Seed-1 training manifest",
    ),
    (
        SRC_SEED1_COMPLETION_MANIFEST,
        "Seed-1 completion manifest",
    ),
    (
        SRC_SEED1_QC_SUMMARY,
        "Seed-1 QC summary",
    ),
    (
        SRC_MULTISEED_PLAN,
        "Frozen multi-seed plan",
    ),
    (
        SRC_SEED2_CONFIG,
        "Frozen Seed-2 config",
    ),
    (
        SRC_SEED3_CONFIG,
        "Frozen Seed-3 config",
    ),
    (
        SRC_DECODER,
        "Validated decoder",
    ),
    (
        SRC_TRAINER,
        "Validated trainer",
    ),
    (
        SRC_LAUNCHER,
        "Validated launcher",
    ),
]


for path, label in required_inputs:

    require(
        path,
        label,
    )

    print(
        f"PASS  {label}"
    )


# ============================================================
# 8. EOD STATE CONSISTENCY
# ============================================================

state = load_json(
    SOURCE_EOD_STATE
)

state_top = load_json(
    TOPLEVEL_STATE
)


if state != state_top:

    raise RuntimeError(
        "Top-level EOD state differs from the "
        "state stored inside the handoff."
    )


critical = state.get(
    "critical_hashes",
    {}
)


required_critical_keys = [
    "seed1_terminal",
    "seed1_training_manifest",
    "seed1_completion_manifest",
    "seed1_qc_summary",
    "multiseed_plan",
    "seed2_config",
    "seed3_config",
    "recipe_freeze_zip",
    "decoder",
]


for key in required_critical_keys:

    if key not in critical:

        raise RuntimeError(
            f"EOD_STATE missing critical hash: {key}"
        )


if (
    critical[
        "seed1_terminal"
    ]
    != EXPECTED_SEED1_TERMINAL_SHA
):

    raise RuntimeError(
        "EOD_STATE Seed-1 terminal identity mismatch."
    )


if (
    critical[
        "recipe_freeze_zip"
    ]
    != EXPECTED_ORIGINAL_RECIPE_ZIP_SHA
):

    raise RuntimeError(
        "EOD_STATE original recipe ZIP provenance mismatch."
    )


if (
    critical[
        "decoder"
    ]
    != EXPECTED_DECODER_SHA
):

    raise RuntimeError(
        "EOD_STATE decoder identity mismatch."
    )


print()
print(
    "EOD state copies identical: PASS"
)

print(
    "Historical original recipe ZIP SHA provenance: PASS"
)


# ============================================================
# 9. VERIFY ALL SURVIVING CRITICAL SOURCE FILES
# ============================================================

source_hash_checks = {
    "seed1_terminal":
        SRC_SEED1_TERMINAL,

    "seed1_training_manifest":
        SRC_SEED1_TRAINING_MANIFEST,

    "seed1_completion_manifest":
        SRC_SEED1_COMPLETION_MANIFEST,

    "seed1_qc_summary":
        SRC_SEED1_QC_SUMMARY,

    "multiseed_plan":
        SRC_MULTISEED_PLAN,

    "seed2_config":
        SRC_SEED2_CONFIG,

    "seed3_config":
        SRC_SEED3_CONFIG,

    "decoder":
        SRC_DECODER,
}


print()
print(
    "============================================================"
)

print(
    "CRITICAL SOURCE HASH AUDIT"
)

print(
    "============================================================"
)


verified_hashes = {}


for key, path in source_hash_checks.items():

    actual = sha256_file(
        path
    )

    expected = critical[
        key
    ]


    if actual != expected:

        raise RuntimeError(
            f"Critical source SHA failure: {key}\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}\n"
            f"Path:     {path}"
        )


    verified_hashes[
        key
    ] = actual


    print(
        f"PASS  {key}"
    )


# ============================================================
# 10. VERIFY THE TWO EXTRACTED FREEZE TREES
#
# Kaggle unpacked the renamed copy of the original archive.
# Those two trees must be file-for-file identical.
# ============================================================

print()
print(
    "============================================================"
)

print(
    "EXTRACTED RECIPE-FREEZE DUPLICATE AUDIT"
)

print(
    "============================================================"
)


primary_freeze_hashes = tree_hash_map(
    SRC_FREEZE
)

archive_freeze_hashes = tree_hash_map(
    SRC_FREEZE_ARCHIVE_COPY
)


if not primary_freeze_hashes:

    raise RuntimeError(
        "Primary freeze tree contains no files."
    )


if (
    primary_freeze_hashes
    != archive_freeze_hashes
):

    primary_keys = set(
        primary_freeze_hashes
    )

    archive_keys = set(
        archive_freeze_hashes
    )


    only_primary = sorted(
        primary_keys
        - archive_keys
    )

    only_archive = sorted(
        archive_keys
        - primary_keys
    )

    changed = sorted(
        key
        for key in (
            primary_keys
            & archive_keys
        )
        if (
            primary_freeze_hashes[
                key
            ]
            != archive_freeze_hashes[
                key
            ]
        )
    )


    raise RuntimeError(
        "The two extracted freeze trees are not identical.\n"
        f"Only primary: {only_primary[:10]}\n"
        f"Only archive: {only_archive[:10]}\n"
        f"Changed:      {changed[:10]}"
    )


print(
    "Freeze trees file-for-file identical: PASS"
)

print(
    "Freeze files verified:",
    len(
        primary_freeze_hashes
    ),
)


# ============================================================
# 11. VERIFY SCIENTIFIC PLAN BEFORE RESTORE
# ============================================================

plan = load_json(
    SRC_MULTISEED_PLAN
)

seed1_cfg = load_json(
    SRC_SEED1_CONFIG
)

seed2_cfg = load_json(
    SRC_SEED2_CONFIG
)

seed3_cfg = load_json(
    SRC_SEED3_CONFIG
)

training_manifest = load_json(
    SRC_SEED1_TRAINING_MANIFEST
)

completion_manifest = load_json(
    SRC_SEED1_COMPLETION_MANIFEST
)

qc_summary = load_json(
    SRC_SEED1_QC_SUMMARY
)


if int(
    plan[
        "frozen_training_recipe"
    ][
        "optimizer_updates"
    ]
) != EXPECTED_TARGET_UPDATES:

    raise RuntimeError(
        "Frozen final training budget is not 2750."
    )


if int(
    seed1_cfg[
        "seed"
    ]
) != EXPECTED_SEED1:

    raise RuntimeError(
        "Seed-1 config identity failure."
    )


if int(
    seed2_cfg[
        "seed"
    ]
) != EXPECTED_SEED2:

    raise RuntimeError(
        "Seed-2 config identity failure."
    )


if int(
    seed3_cfg[
        "seed"
    ]
) != EXPECTED_SEED3:

    raise RuntimeError(
        "Seed-3 config identity failure."
    )


if (
    seed2_cfg[
        "experiment_name"
    ]
    != "dasonet_final_seed02_s901395089"
):

    raise RuntimeError(
        "Seed-2 experiment name failure."
    )


pcc_center = (
    seed2_cfg
    .get(
        "loss",
        {}
    )
    .get(
        "pcc_center"
    )
)


if (
    pcc_center is not None
    and bool(
        pcc_center.get(
            "enabled",
            False,
        )
    )
):

    raise RuntimeError(
        "Seed-2 PCC-center unexpectedly enabled."
    )


if int(
    training_manifest[
        "successful_optimizer_updates"
    ]
) != EXPECTED_TARGET_UPDATES:

    raise RuntimeError(
        "Seed-1 final training manifest is not at 2750."
    )


if int(
    completion_manifest[
        "final_optimizer_step"
    ]
) != EXPECTED_TARGET_UPDATES:

    raise RuntimeError(
        "Seed-1 completion manifest is not at 2750."
    )


if not all(
    qc_summary[
        "sanity_flags"
    ].values()
):

    raise RuntimeError(
        "Seed-1 QC contains a failed sanity flag."
    )


print()
print(
    "Frozen training budget: 2750 — PASS"
)

print(
    "Seed 1 identity: PASS"
)

print(
    "Seed 2 identity: PASS"
)

print(
    "Seed 3 identity: PASS"
)

print(
    "Seed-2 PCC-center disabled: PASS"
)

print(
    "Seed-1 final QC sanity flags: PASS"
)


# ============================================================
# 12. VERIFY PROJECT DATASET BASIC IDENTITY
# ============================================================

project_required = [
    PROJECT_ROOT
    / "data"
    / "mp_morph"
    / "v1"
    / "morphology_manifest_v1.csv",

    PROJECT_ROOT
    / "data"
    / "fruit_bg"
    / "v1_final"
    / "fruitbg_rgb_v1_manifest.csv",

    PROJECT_ROOT
    / "configs"
    / "renderer_v3_1.json",

    PROJECT_ROOT
    / "src"
    / "dasonet"
    / "synth"
    / "renderer.py",
]


for path in project_required:

    require(
        path,
        "Frozen project component",
    )


print(
    "Frozen project critical components: PASS"
)


# ============================================================
# 13. GPU RUNTIME AUDIT
# ============================================================

if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA is unavailable. "
        "Select GPU accelerator before continuing."
    )


gpu_count = torch.cuda.device_count()


if gpu_count != 2:

    raise RuntimeError(
        f"Expected 2 GPUs; found {gpu_count}."
    )


gpu_names = [
    torch.cuda.get_device_name(
        i
    )
    for i in range(
        gpu_count
    )
]


if not all(
    "T4" in name
    for name in gpu_names
):

    raise RuntimeError(
        "Final multi-seed protocol requires 2x Tesla T4.\n"
        f"Detected: {gpu_names}"
    )


print(
    "GPU runtime: 2x Tesla T4 — PASS"
)


# ============================================================
# 14. FRESH SEED-2 SAFETY CHECK BEFORE COPYING
# ============================================================

SEED2_OUTPUT = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed02_s901395089"
)


if SEED2_OUTPUT.exists():

    raise RuntimeError(
        "Seed-2 output directory already exists:\n"
        f"{SEED2_OUTPUT}\n\n"
        "STOP. Fresh initialization cannot be assumed."
    )


print(
    "Seed-2 output absent before restore: PASS"
)


# ============================================================
# 15. RESTORE TO /kaggle/working
# ============================================================

print()
print(
    "============================================================"
)

print(
    "RESTORING HANDOFF TO /kaggle/working"
)

print(
    "============================================================"
)


tree_copies = [
    (
        SRC_CODE,
        DST_CODE,
        "Validated code tree",
    ),
    (
        SRC_MULTISEED,
        DST_MULTISEED,
        "Frozen multi-seed plan",
    ),
    (
        SRC_FREEZE,
        DST_FREEZE,
        "Frozen final-training recipe",
    ),
    (
        SRC_SEED1_RUN,
        DST_SEED1_RUN,
        "Final Seed-1 run",
    ),
    (
        SRC_SEED1_QC,
        DST_SEED1_QC,
        "Final Seed-1 QC",
    ),
    (
        SRC_DURATION,
        DST_DURATION,
        "Duration-selection evidence",
    ),
]


for src, dst, label in tree_copies:

    replace_tree(
        src,
        dst,
    )

    print(
        f"PASS  restored {label}"
    )


file_copies = [
    (
        SRC_BASE_CONFIG,
        DST_BASE_CONFIG,
        "Source training config",
    ),
    (
        SRC_CURVE_SUMMARY,
        DST_CURVE_SUMMARY,
        "Learning-curve summary",
    ),
]


if SRC_PREFLIGHT.exists():

    file_copies.append(
        (
            SRC_PREFLIGHT,
            DST_PREFLIGHT,
            "Multi-seed preflight",
        )
    )


for src, dst, label in file_copies:

    copy_file(
        src,
        dst,
    )

    print(
        f"PASS  restored {label}"
    )


# ============================================================
# 16. POST-RESTORE CRITICAL AUDIT
# ============================================================

DST_SEED1_TERMINAL = (
    DST_SEED1_RUN
    / "snapshots"
    / "epoch_011_opt_002750.pt"
)

DST_PLAN = (
    DST_MULTISEED
    / "FINAL_MULTISEED_PLAN_v1.json"
)

DST_SEED2_CONFIG = (
    DST_MULTISEED
    / "configs"
    / "final_seed02_s901395089.json"
)

DST_SEED3_CONFIG = (
    DST_MULTISEED
    / "configs"
    / "final_seed03_s252916394.json"
)

DST_DECODER = (
    DST_CODE
    / "src"
    / "dasonet"
    / "model"
    / "decoding.py"
)

DST_TRAINER = (
    DST_CODE
    / "scripts"
    / "train_phase5b.py"
)

DST_LAUNCHER = (
    DST_CODE
    / "scripts"
    / "launch_phase5c_checkpointed_baseline.py"
)


post_checks = {
    "seed1_terminal":
        DST_SEED1_TERMINAL,

    "multiseed_plan":
        DST_PLAN,

    "seed2_config":
        DST_SEED2_CONFIG,

    "seed3_config":
        DST_SEED3_CONFIG,

    "decoder":
        DST_DECODER,
}


for key, path in post_checks.items():

    require(
        path,
        f"Restored {key}",
    )


    actual = sha256_file(
        path
    )

    expected = critical[
        key
    ]


    if actual != expected:

        raise RuntimeError(
            f"POST-RESTORE SHA FAILURE: {key}\n"
            f"Expected: {expected}\n"
            f"Actual:   {actual}"
        )


print()
print(
    "Restored Seed-1 terminal SHA: PASS"
)

print(
    "Restored multi-seed plan SHA: PASS"
)

print(
    "Restored Seed-2 config SHA: PASS"
)

print(
    "Restored Seed-3 config SHA: PASS"
)

print(
    "Restored decoder SHA: PASS"
)


# ============================================================
# 17. VERIFY SEED-2 IS STILL FRESH
# ============================================================

if SEED2_OUTPUT.exists():

    raise RuntimeError(
        "Seed-2 output appeared during restore unexpectedly."
    )


restored_seed2 = load_json(
    DST_SEED2_CONFIG
)


if int(
    restored_seed2[
        "seed"
    ]
) != EXPECTED_SEED2:

    raise RuntimeError(
        "Restored Seed-2 RNG seed mismatch."
    )


if (
    restored_seed2[
        "experiment_name"
    ]
    != "dasonet_final_seed02_s901395089"
):

    raise RuntimeError(
        "Restored Seed-2 experiment mismatch."
    )


print(
    "Seed-2 fresh-output condition: PASS"
)

print(
    "Seed-2 RNG seed 901395089: PASS"
)

print(
    "Seed-2 resume policy: NONE"
)


# ============================================================
# 18. WRITE RESTORE AUDIT RECORD
# ============================================================

restore_audit = {
    "audit_name":
        "DASO-Net-AFTER-SEED1-RESTORE-AUDIT-v2",

    "source_handoff_root":
        str(
            HANDOFF_ROOT
        ),

    "project_root":
        str(
            PROJECT_ROOT
        ),

    "kaggle_archive_behavior":
        {
            "outer_sanitized_zip_auto_extracted":
                True,

            "renamed_inner_recipe_zip_auto_extracted":
                True,

            "original_recipe_zip_reconstructed":
                False,

            "reason":
                (
                    "Reconstructing a ZIP would not preserve the "
                    "historical byte-level SHA256. Original ZIP SHA "
                    "is retained as provenance in EOD_STATE instead."
                ),

            "historical_original_recipe_zip_sha256":
                EXPECTED_ORIGINAL_RECIPE_ZIP_SHA,

            "two_extracted_recipe_trees_identical":
                True,

            "freeze_file_count":
                len(
                    primary_freeze_hashes
                ),
        },

    "seed1":
        {
            "rng_seed":
                EXPECTED_SEED1,

            "successful_optimizer_updates":
                2750,

            "terminal_sha256":
                sha256_file(
                    DST_SEED1_TERMINAL
                ),

            "qc_restored":
                True,
        },

    "next_seed":
        {
            "seed_index":
                2,

            "rng_seed":
                EXPECTED_SEED2,

            "experiment_name":
                "dasonet_final_seed02_s901395089",

            "resume":
                "none",

            "fresh_output_directory_absent":
                True,

            "target_successful_optimizer_updates":
                2750,
        },

    "runtime":
        {
            "gpu_count":
                gpu_count,

            "gpu_names":
                gpu_names,
        },

    "critical_hashes_verified":
        verified_hashes,
}


RESTORE_AUDIT_JSON.write_text(
    json.dumps(
        restore_audit,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 19. FINAL REPORT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "AFTER-SEED1 RESTORE COMPLETE"
)

print(
    "============================================================"
)

print(
    "Seed 1:"
)

print(
    "  status:                  COMPLETE"
)

print(
    "  successful updates:      2750"
)

print(
    "  terminal SHA:"
)

print(
    " ",
    sha256_file(
        DST_SEED1_TERMINAL
    ),
)

print()

print(
    "Seed 2:"
)

print(
    "  RNG seed:                901395089"
)

print(
    "  initialization:          FRESH"
)

print(
    "  resume:                  NONE"
)

print(
    "  target successful steps: 2750"
)

print(
    "  output absent:           YES"
)

print()

print(
    "Runtime:"
)

print(
    "  GPUs:                    ",
    gpu_names,
)

print()

print(
    "Restore audit:"
)

print(
    RESTORE_AUDIT_JSON
)

print()

print(
    "============================================================"
)

print(
    "DASO-NET AFTER-SEED1 RESTORE + SEED-2 READINESS: PASS"
)

print(
    "============================================================"
)

print(
    "NO TRAINING WAS STARTED."
)

print(
    "STOP HERE."
)

print(
    "Paste this final output before launching Final Seed 2."
)

DASO-Net — AFTER-SEED1 RESTORE + SEED-2 READINESS v2
PASS  Frozen project dataset
PASS  EOD Kaggle dataset
PASS  Extracted EOD handoff root
PASS  Extracted EOD workspace
PASS  EOD state
PASS  EOD manifest
PASS  Top-level EOD state
PASS  Validated code tree
PASS  Frozen multi-seed directory
PASS  Primary extracted freeze tree
PASS  Archive-derived freeze tree
PASS  Seed-1 terminal model
PASS  Seed-1 training manifest
PASS  Seed-1 completion manifest
PASS  Seed-1 QC summary
PASS  Frozen multi-seed plan
PASS  Frozen Seed-2 config
PASS  Frozen Seed-3 config
PASS  Validated decoder
PASS  Validated trainer
PASS  Validated launcher

EOD state copies identical: PASS
Historical original recipe ZIP SHA provenance: PASS

CRITICAL SOURCE HASH AUDIT
PASS  seed1_terminal
PASS  seed1_training_manifest
PASS  seed1_completion_manifest
PASS  seed1_qc_summary
PASS  multiseed_plan
PASS  seed2_config
PASS  seed3_config
PASS  decoder

EXTRACTED RECIPE-FREEZE DUPLICATE AUDIT
Freeze trees file-for-file identi

In [3]:
# ============================================================
# DASO-Net — FINAL MULTI-SEED TRAINING
# FINAL SEED 2 / 3
#
# RNG seed:
#   901395089
#
# FROZEN SCIENTIFIC RECIPE
# ------------------------
# fresh initialization
# target = EXACTLY 2750 successful optimizer updates
# native tile = 1024
# 2 x Tesla T4
# synthetic batch/GPU = 1
# anchor pair batch/GPU = 2
# gradient accumulation = 4
# effective synthetic batch = 8
# synthetic workers/GPU = 2
# anchor workers/GPU = 1
# LR = 3e-4
# warmup = 50 successful optimizer updates
# constant post-warmup LR
# AMP FP16 initial scale = 32
# original PCC retained
# CRPA retained
# PCC-center disabled
#
# IMPORTANT
# ---------
# - Initial run is FRESH: --resume none
# - No early stopping
# - best.pt is NOT used for final reporting
# - terminal model = checkpoint at exactly 2750 successful
#   optimizer updates
# - if AMP skips occur, bounded compensation is performed
#   automatically
# - external data are NOT used
#
# After PASS:
#   STOP. Do not start Seed 3.
# ============================================================

from __future__ import annotations

import hashlib
import json
import shutil
import subprocess
import time
from pathlib import Path

import torch


# ============================================================
# 1. FROZEN CONSTANTS
# ============================================================

SEED_INDEX = 2

RNG_SEED = 901395089

EXPERIMENT = (
    "dasonet_final_seed02_s901395089"
)

TARGET_OPTIMIZER_STEP = 2750

INITIAL_EPOCHS = 11

INITIAL_MICROSTEPS_PER_EPOCH = 1000

ACCUM = 4

EXPECTED_UPDATES_PER_FULL_BLOCK = (
    INITIAL_MICROSTEPS_PER_EPOCH // ACCUM
)

EXPECTED_INITIAL_UPDATE_ATTEMPTS = (
    INITIAL_EPOCHS
    * EXPECTED_UPDATES_PER_FULL_BLOCK
)

EXPECTED_SEED1_SHA = (
    "e4c4b79ad4b8c6e18ba2b1e3d6cbe796"
    "c9cc301bd6fcc7c596fc290ad3a7e81d"
)

EXPECTED_DECODER_SHA = (
    "9799eb1be568e3d6e0ce82b7c99df7a0"
    "ed05d271cd44ff24b612526cc18537d1"
)


# ============================================================
# 2. PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

WORKING = Path(
    "/kaggle/working"
)

CODE_ROOT = (
    WORKING
    / "DASO-Net-Phase5B-KaggleTraining"
)

TRAINER = (
    CODE_ROOT
    / "scripts"
    / "train_phase5b.py"
)

LAUNCHER = (
    CODE_ROOT
    / "scripts"
    / "launch_phase5c_checkpointed_baseline.py"
)

DECODER = (
    CODE_ROOT
    / "src"
    / "dasonet"
    / "model"
    / "decoding.py"
)

PLAN_ROOT = (
    WORKING
    / "DASO-Net-FinalMultiSeed-v1"
)

PLAN_JSON = (
    PLAN_ROOT
    / "FINAL_MULTISEED_PLAN_v1.json"
)

CONFIG = (
    PLAN_ROOT
    / "configs"
    / "final_seed02_s901395089.json"
)

RESTORE_AUDIT = (
    WORKING
    / "DASO-Net-AFTER-SEED1-RESTORE-AUDIT-v2.json"
)

SEED1_TERMINAL = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed01_s1782656963"
    / "snapshots"
    / "epoch_011_opt_002750.pt"
)

RUN_ROOT = (
    WORKING
    / "dasonet_phase5b"
    / EXPERIMENT
)

SNAPSHOT_DIR = (
    RUN_ROOT
    / "snapshots"
)

RUN_STATUS = (
    RUN_ROOT
    / "run_status.json"
)

RESOLVED_CONFIG = (
    RUN_ROOT
    / "resolved_config.json"
)

HISTORY = (
    RUN_ROOT
    / "history.csv"
)

INITIAL_STATUS_COPY = (
    RUN_ROOT
    / "run_status_initial_11blocks.json"
)

INITIAL_RESOLVED_COPY = (
    RUN_ROOT
    / "resolved_config_initial_11blocks.json"
)

INITIAL_HISTORY_COPY = (
    RUN_ROOT
    / "history_initial_11blocks.csv"
)

EXACT_UPDATE_MANIFEST = (
    RUN_ROOT
    / "FINAL_SEED02_EXACT_UPDATE_COMPLETION.json"
)

FINAL_TRAINING_MANIFEST = (
    RUN_ROOT
    / "FINAL_SEED02_TRAINING_MANIFEST.json"
)


# ============================================================
# 3. HELPERS
# ============================================================

def require(
    path: Path,
    label: str,
) -> None:

    if not path.exists():

        raise RuntimeError(
            f"{label} missing:\n{path}"
        )


def sha256_file(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):
            h.update(
                block
            )

    return h.hexdigest()


def load_json(
    path: Path,
):

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def trusted_load(
    path: Path,
):

    return torch.load(
        path,
        map_location="cpu",
        weights_only=False,
    )


def snapshot_records():
    """
    Read every chronological epoch snapshot and return
    metadata sorted by epoch then optimizer step.
    """

    records = []

    if not SNAPSHOT_DIR.exists():

        return records


    for path in SNAPSHOT_DIR.glob(
        "epoch_*_opt_*.pt"
    ):

        payload = trusted_load(
            path
        )

        records.append(
            {
                "path":
                    path,

                "epoch":
                    int(
                        payload.get(
                            "epoch",
                            -1,
                        )
                    ),

                "optimizer_step":
                    int(
                        payload.get(
                            "optimizer_step",
                            -1,
                        )
                    ),

                "sha256":
                    sha256_file(
                        path
                    ),
            }
        )


    records.sort(
        key=lambda x: (
            x[
                "epoch"
            ],
            x[
                "optimizer_step"
            ],
        )
    )

    return records


def latest_snapshot():
    """
    Return the chronologically latest snapshot.
    """

    records = snapshot_records()

    if not records:

        raise RuntimeError(
            "No checkpoint snapshots found."
        )

    return records[-1]


def verify_constant_lr(
    checkpoint: Path,
) -> list[float]:

    payload = trusted_load(
        checkpoint
    )

    optimizer_state = payload.get(
        "optimizer_state_dict",
        {}
    )

    groups = optimizer_state.get(
        "param_groups",
        []
    )


    lrs = [
        float(
            group[
                "lr"
            ]
        )
        for group in groups
        if "lr" in group
    ]


    if not lrs:

        raise RuntimeError(
            "Could not read optimizer learning rate "
            "from checkpoint."
        )


    for lr in lrs:

        if abs(
            lr
            - 3e-4
        ) > 1e-10:

            raise RuntimeError(
                "Unexpected final/completion learning rate.\n"
                f"Observed: {lr}"
            )


    return lrs


# ============================================================
# 4. PREFLIGHT
# ============================================================

print(
    "============================================================"
)

print(
    "DASO-Net — FINAL SEED 2 / 3 PREFLIGHT"
)

print(
    "============================================================"
)


for path, label in [
    (
        PROJECT_ROOT,
        "Frozen project dataset",
    ),
    (
        CODE_ROOT,
        "Restored validated code tree",
    ),
    (
        TRAINER,
        "Validated trainer",
    ),
    (
        LAUNCHER,
        "Validated checkpoint launcher",
    ),
    (
        DECODER,
        "Validated decoder",
    ),
    (
        PLAN_JSON,
        "Frozen multi-seed plan",
    ),
    (
        CONFIG,
        "Frozen Seed-2 config",
    ),
    (
        RESTORE_AUDIT,
        "After-Seed1 restore audit",
    ),
    (
        SEED1_TERMINAL,
        "Final Seed-1 terminal checkpoint",
    ),
]:

    require(
        path,
        label,
    )

    print(
        f"PASS  {label}"
    )


# ============================================================
# 5. VERIFY RESTORED STATE
# ============================================================

restore = load_json(
    RESTORE_AUDIT
)


if (
    restore[
        "next_seed"
    ][
        "rng_seed"
    ]
    != RNG_SEED
):

    raise RuntimeError(
        "Restore audit does not designate Seed 2 correctly."
    )


if (
    restore[
        "next_seed"
    ][
        "resume"
    ]
    != "none"
):

    raise RuntimeError(
        "Restore audit Seed-2 resume policy is not NONE."
    )


if int(
    restore[
        "next_seed"
    ][
        "target_successful_optimizer_updates"
    ]
) != TARGET_OPTIMIZER_STEP:

    raise RuntimeError(
        "Restore audit target is not 2750."
    )


if (
    sha256_file(
        SEED1_TERMINAL
    )
    != EXPECTED_SEED1_SHA
):

    raise RuntimeError(
        "Seed-1 terminal checkpoint changed after restore."
    )


if (
    sha256_file(
        DECODER
    )
    != EXPECTED_DECODER_SHA
):

    raise RuntimeError(
        "Validated decoder SHA mismatch."
    )


print()
print(
    "After-Seed1 restore state: PASS"
)

print(
    "Seed-1 terminal remains intact: PASS"
)

print(
    "Validated decoder SHA: PASS"
)


# ============================================================
# 6. VERIFY FROZEN MULTI-SEED PLAN + SEED-2 CONFIG
# ============================================================

plan = load_json(
    PLAN_JSON
)

cfg = load_json(
    CONFIG
)


if (
    plan.get(
        "status"
    )
    != "FROZEN_BEFORE_FINAL_TRAINING"
):

    raise RuntimeError(
        "Multi-seed plan status is not frozen."
    )


if int(
    plan[
        "frozen_training_recipe"
    ][
        "optimizer_updates"
    ]
) != TARGET_OPTIMIZER_STEP:

    raise RuntimeError(
        "Frozen training budget is not 2750."
    )


planned_seed2 = None


for item in plan[
    "final_seed_configs"
]:

    if int(
        item[
            "seed_index"
        ]
    ) == SEED_INDEX:

        planned_seed2 = item
        break


if planned_seed2 is None:

    raise RuntimeError(
        "Seed-2 entry missing from frozen multi-seed plan."
    )


if int(
    planned_seed2[
        "seed"
    ]
) != RNG_SEED:

    raise RuntimeError(
        "Seed-2 RNG seed mismatch in frozen plan."
    )


if (
    planned_seed2[
        "experiment_name"
    ]
    != EXPERIMENT
):

    raise RuntimeError(
        "Seed-2 experiment mismatch in frozen plan."
    )


actual_config_sha = sha256_file(
    CONFIG
)


if (
    actual_config_sha
    != planned_seed2[
        "config_sha256"
    ]
):

    raise RuntimeError(
        "Seed-2 config SHA mismatch.\n"
        f"Expected: {planned_seed2['config_sha256']}\n"
        f"Actual:   {actual_config_sha}"
    )


if int(
    cfg[
        "seed"
    ]
) != RNG_SEED:

    raise RuntimeError(
        "Seed-2 config RNG mismatch."
    )


if (
    cfg[
        "experiment_name"
    ]
    != EXPERIMENT
):

    raise RuntimeError(
        "Seed-2 config experiment mismatch."
    )


pcc_center = (
    cfg
    .get(
        "loss",
        {}
    )
    .get(
        "pcc_center"
    )
)


if (
    pcc_center is not None
    and bool(
        pcc_center.get(
            "enabled",
            False,
        )
    )
):

    raise RuntimeError(
        "PCC-center unexpectedly enabled."
    )


print(
    "Frozen multi-seed plan: PASS"
)

print(
    "Seed-2 config SHA: PASS"
)

print(
    "Seed-2 RNG seed: 901395089 — PASS"
)

print(
    "PCC-center disabled: PASS"
)


# ============================================================
# 7. FRESH-INITIALIZATION SAFETY
# ============================================================

if RUN_ROOT.exists():

    raise RuntimeError(
        "FINAL SEED-2 OUTPUT DIRECTORY ALREADY EXISTS:\n"
        f"{RUN_ROOT}\n\n"
        "Refusing to start. Fresh initialization cannot "
        "be guaranteed.\n"
        "Do NOT delete anything manually; inspect first."
    )


print()
print(
    "Fresh initialization condition: PASS"
)

print(
    "Seed-2 output directory is absent."
)


# ============================================================
# 8. GPU AUDIT
# ============================================================

if not torch.cuda.is_available():

    raise RuntimeError(
        "CUDA unavailable."
    )


if torch.cuda.device_count() != 2:

    raise RuntimeError(
        f"Expected exactly 2 GPUs; "
        f"found {torch.cuda.device_count()}."
    )


gpu_names = [
    torch.cuda.get_device_name(
        i
    )
    for i in range(2)
]


if not all(
    "T4" in name
    for name in gpu_names
):

    raise RuntimeError(
        "Frozen runtime requires 2x Tesla T4.\n"
        f"Detected: {gpu_names}"
    )


print(
    "GPU runtime: 2x Tesla T4 — PASS"
)


# ============================================================
# 9. TRAINING CONTRACT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "FINAL SEED 2 TRAINING CONTRACT"
)

print(
    "============================================================"
)

print(
    "Seed index:                  2 / 3"
)

print(
    "RNG seed:                    901395089"
)

print(
    "Initialization:              FRESH"
)

print(
    "Initial resume:              NONE"
)

print(
    "Full training blocks:        11"
)

print(
    "Microsteps/rank/block:       1000"
)

print(
    "Gradient accumulation:       4"
)

print(
    "Possible updates/block:      250"
)

print(
    "Initial update attempts:     2750"
)

print(
    "Final SUCCESSFUL updates:    EXACTLY 2750"
)

print(
    "Native tile:                 1024"
)

print(
    "GPUs:                        2 x T4"
)

print(
    "Synthetic batch/GPU:         1"
)

print(
    "Anchor pair batch/GPU:       2"
)

print(
    "Effective synthetic batch:   8"
)

print(
    "Synthetic workers/GPU:       2"
)

print(
    "Anchor workers/GPU:          1"
)

print(
    "Learning rate:               3e-4"
)

print(
    "Warmup successful updates:   50"
)

print(
    "Post-warmup LR:              CONSTANT"
)

print(
    "AMP:                         FP16"
)

print(
    "AMP initial scale:           32"
)

print(
    "Original PCC:                RETAINED"
)

print(
    "CRPA:                        RETAINED"
)

print(
    "PCC-center:                  DISABLED"
)

print(
    "Early stopping:              NO"
)

print(
    "Final checkpoint policy:     exact step 2750"
)

print(
    "best.pt final reporting:     NO"
)

print(
    "External data tuning:        NONE"
)

print()


# ============================================================
# 10. FRESH INITIAL 11-BLOCK TRAINING
# ============================================================

initial_cmd = [
    "torchrun",

    "--standalone",

    "--nproc_per_node=2",

    str(
        LAUNCHER
    ),

    "--project-root",
    str(
        PROJECT_ROOT
    ),

    "--config",
    str(
        CONFIG
    ),

    "--output-root",
    str(
        WORKING
    ),

    "--epochs",
    str(
        INITIAL_EPOCHS
    ),

    "--max-train-steps",
    str(
        INITIAL_MICROSTEPS_PER_EPOCH
    ),

    "--max-val-steps",
    "100",

    "--accum",
    str(
        ACCUM
    ),

    "--per-rank-batch",
    "1",

    "--anchor-batch",
    "2",

    "--num-workers",
    "2",

    "--log-every",
    "100",

    "--resume",
    "none",
]


print(
    "============================================================"
)

print(
    "STARTING FINAL SEED 2 — FRESH RUN"
)

print(
    "============================================================"
)

print(
    " ".join(
        initial_cmd
    )
)

print()


overall_start = time.time()

initial_start = time.time()


subprocess.run(
    initial_cmd,
    check=True,
)


initial_elapsed = (
    time.time()
    - initial_start
)


# ============================================================
# 11. PRESERVE INITIAL FULL-RUN METADATA BEFORE ANY COMPENSATION
# ============================================================

for path, label in [
    (
        RUN_ROOT,
        "Seed-2 run directory",
    ),
    (
        SNAPSHOT_DIR,
        "Seed-2 snapshots",
    ),
    (
        RUN_STATUS,
        "Seed-2 run status",
    ),
    (
        RESOLVED_CONFIG,
        "Seed-2 resolved config",
    ),
    (
        HISTORY,
        "Seed-2 history",
    ),
]:

    require(
        path,
        label,
    )


shutil.copy2(
    RUN_STATUS,
    INITIAL_STATUS_COPY,
)

shutil.copy2(
    RESOLVED_CONFIG,
    INITIAL_RESOLVED_COPY,
)

shutil.copy2(
    HISTORY,
    INITIAL_HISTORY_COPY,
)


initial_status = load_json(
    INITIAL_STATUS_COPY
)


initial_records = snapshot_records()


if len(
    initial_records
) != INITIAL_EPOCHS:

    raise RuntimeError(
        "Expected exactly 11 snapshots after the initial "
        f"Seed-2 run; found {len(initial_records)}."
    )


latest = initial_records[-1]


initial_terminal_checkpoint = (
    latest[
        "path"
    ]
)

initial_terminal_epoch = int(
    latest[
        "epoch"
    ]
)

initial_successful_updates = int(
    latest[
        "optimizer_step"
    ]
)


if initial_terminal_epoch != 10:

    raise RuntimeError(
        "Expected initial terminal epoch 10; "
        f"found {initial_terminal_epoch}."
    )


if (
    initial_successful_updates
    > TARGET_OPTIMIZER_STEP
):

    raise RuntimeError(
        "CRITICAL: initial run exceeded frozen 2750 target."
    )


initial_inferred_amp_skips = (
    EXPECTED_INITIAL_UPDATE_ATTEMPTS
    - initial_successful_updates
)


if initial_inferred_amp_skips < 0:

    raise RuntimeError(
        "Invalid initial AMP-skip inference."
    )


reported_initial_amp_skips = int(
    initial_status.get(
        "amp_skipped_optimizer_steps",
        -1,
    )
)


if (
    reported_initial_amp_skips
    != initial_inferred_amp_skips
):

    raise RuntimeError(
        "Initial Seed-2 AMP-skip accounting mismatch.\n"
        f"Inferred from optimizer step: "
        f"{initial_inferred_amp_skips}\n"
        f"run_status reports:          "
        f"{reported_initial_amp_skips}"
    )


print()
print(
    "============================================================"
)

print(
    "INITIAL FINAL-SEED-2 RUN COMPLETE"
)

print(
    "============================================================"
)

print(
    "Successful optimizer updates:",
    initial_successful_updates,
)

print(
    "AMP skipped updates:",
    initial_inferred_amp_skips,
)

print(
    "Initial terminal epoch:",
    initial_terminal_epoch,
)

print(
    "Initial terminal checkpoint:"
)

print(
    initial_terminal_checkpoint
)

print()


# ============================================================
# 12. EXACT-UPDATE COMPENSATION
#
# If the initial run suffered AMP skips, request only:
#
#   remaining_updates * ACCUM
#
# microsteps.
#
# If another AMP skip occurs, repeat with only the remaining
# count.
#
# This cannot intentionally exceed 2750.
# ============================================================

current_checkpoint = (
    initial_terminal_checkpoint
)

current_epoch = (
    initial_terminal_epoch
)

current_step = (
    initial_successful_updates
)

completion_iterations = []

MAX_COMPLETION_ITERATIONS = 10


while (
    current_step
    < TARGET_OPTIMIZER_STEP
):

    if (
        len(
            completion_iterations
        )
        >= MAX_COMPLETION_ITERATIONS
    ):

        raise RuntimeError(
            "Too many Seed-2 AMP compensation iterations.\n"
            "STOP and inspect."
        )


    missing_updates = (
        TARGET_OPTIMIZER_STEP
        - current_step
    )


    requested_microsteps = (
        missing_updates
        * ACCUM
    )


    next_epoch = (
        current_epoch
        + 1
    )


    epochs_cli = (
        next_epoch
        + 1
    )


    print()
    print(
        "============================================================"
    )

    print(
        "SEED-2 EXACT-UPDATE COMPENSATION "
        f"ITERATION {len(completion_iterations) + 1}"
    )

    print(
        "============================================================"
    )

    print(
        "Current successful updates:",
        current_step,
    )

    print(
        "Remaining successful updates:",
        missing_updates,
    )

    print(
        "Requested microsteps/rank:",
        requested_microsteps,
    )

    print(
        "Maximum possible new updates:",
        missing_updates,
    )

    print(
        "Resume checkpoint:"
    )

    print(
        current_checkpoint
    )

    print()


    cmd = [
        "torchrun",

        "--standalone",

        "--nproc_per_node=2",

        str(
            LAUNCHER
        ),

        "--project-root",
        str(
            PROJECT_ROOT
        ),

        "--config",
        str(
            CONFIG
        ),

        "--output-root",
        str(
            WORKING
        ),

        "--epochs",
        str(
            epochs_cli
        ),

        "--max-train-steps",
        str(
            requested_microsteps
        ),

        "--max-val-steps",
        "100",

        "--accum",
        str(
            ACCUM
        ),

        "--per-rank-batch",
        "1",

        "--anchor-batch",
        "2",

        "--num-workers",
        "2",

        "--log-every",
        str(
            max(
                1,
                requested_microsteps,
            )
        ),

        "--resume",
        str(
            current_checkpoint
        ),
    ]


    iteration_start = time.time()


    subprocess.run(
        cmd,
        check=True,
    )


    iteration_elapsed = (
        time.time()
        - iteration_start
    )


    records_now = snapshot_records()


    candidates = [
        record
        for record in records_now
        if int(
            record[
                "epoch"
            ]
        ) == next_epoch
    ]


    if len(
        candidates
    ) != 1:

        raise RuntimeError(
            "Expected exactly one snapshot for compensation "
            f"epoch {next_epoch}; found {len(candidates)}."
        )


    new_record = candidates[0]

    new_checkpoint = (
        new_record[
            "path"
        ]
    )

    new_step = int(
        new_record[
            "optimizer_step"
        ]
    )

    new_epoch = int(
        new_record[
            "epoch"
        ]
    )


    if new_step <= current_step:

        raise RuntimeError(
            "Seed-2 compensation made no successful "
            "optimizer progress."
        )


    if new_step > TARGET_OPTIMIZER_STEP:

        raise RuntimeError(
            "CRITICAL: Seed-2 compensation exceeded "
            "the frozen 2750-update target.\n"
            f"Observed: {new_step}"
        )


    successful_new_updates = (
        new_step
        - current_step
    )


    inferred_additional_skips = (
        missing_updates
        - successful_new_updates
    )


    if inferred_additional_skips < 0:

        raise RuntimeError(
            "Invalid compensation AMP-skip inference."
        )


    lr_values = verify_constant_lr(
        new_checkpoint
    )


    completion_iterations.append(
        {
            "iteration":
                len(
                    completion_iterations
                )
                + 1,

            "start_optimizer_step":
                current_step,

            "requested_successful_updates":
                missing_updates,

            "requested_microsteps_per_rank":
                requested_microsteps,

            "successful_updates_observed":
                successful_new_updates,

            "inferred_amp_skips":
                inferred_additional_skips,

            "end_optimizer_step":
                new_step,

            "epoch":
                new_epoch,

            "checkpoint":
                str(
                    new_checkpoint
                ),

            "checkpoint_sha256":
                sha256_file(
                    new_checkpoint
                ),

            "learning_rates":
                lr_values,

            "wall_time_minutes":
                iteration_elapsed
                / 60.0,
        }
    )


    print()
    print(
        "Compensation result:"
    )

    print(
        "  successful new updates:",
        successful_new_updates,
    )

    print(
        "  additional AMP skips:",
        inferred_additional_skips,
    )

    print(
        "  new optimizer step:",
        new_step,
    )

    print(
        "  learning rate:",
        lr_values,
    )


    current_checkpoint = (
        new_checkpoint
    )

    current_epoch = (
        new_epoch
    )

    current_step = (
        new_step
    )


# ============================================================
# 13. EXACT TARGET VERIFICATION
# ============================================================

if current_step != TARGET_OPTIMIZER_STEP:

    raise RuntimeError(
        "Seed 2 did not finish at exactly 2750 "
        "successful optimizer updates."
    )


final_checkpoint = (
    current_checkpoint
)

final_epoch = (
    current_epoch
)

final_sha = sha256_file(
    final_checkpoint
)

final_lrs = verify_constant_lr(
    final_checkpoint
)


additional_amp_skips = sum(
    int(
        item[
            "inferred_amp_skips"
        ]
    )
    for item in completion_iterations
)


total_amp_skips = (
    initial_inferred_amp_skips
    + additional_amp_skips
)


overall_elapsed = (
    time.time()
    - overall_start
)


# ============================================================
# 14. FINAL SNAPSHOT-CHAIN AUDIT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "AUDITING FINAL SEED-2 SNAPSHOT CHAIN"
)

print(
    "============================================================"
)


final_records = snapshot_records()


if not final_records:

    raise RuntimeError(
        "Seed-2 snapshot chain is empty."
    )


previous_epoch = -1
previous_step = -1


snapshot_manifest_records = []


for record in final_records:

    epoch = int(
        record[
            "epoch"
        ]
    )

    step = int(
        record[
            "optimizer_step"
        ]
    )


    if epoch <= previous_epoch:

        raise RuntimeError(
            "Seed-2 snapshot epochs are not strictly increasing."
        )


    if step <= previous_step:

        raise RuntimeError(
            "Seed-2 optimizer steps are not strictly increasing."
        )


    if step > TARGET_OPTIMIZER_STEP:

        raise RuntimeError(
            "Found Seed-2 checkpoint beyond frozen target:\n"
            f"{record['path']}"
        )


    snapshot_manifest_records.append(
        {
            "epoch":
                epoch,

            "optimizer_step":
                step,

            "path":
                str(
                    record[
                        "path"
                    ]
                ),

            "sha256":
                record[
                    "sha256"
                ],
        }
    )


    print(
        f"PASS  "
        f"epoch={epoch:02d}  "
        f"optimizer_step={step:04d}  "
        f"sha={record['sha256'][:18]}..."
    )


    previous_epoch = epoch
    previous_step = step


if int(
    final_records[
        -1
    ][
        "optimizer_step"
    ]
) != TARGET_OPTIMIZER_STEP:

    raise RuntimeError(
        "Last Seed-2 snapshot is not step 2750."
    )


if (
    Path(
        final_records[
            -1
        ][
            "path"
        ]
    )
    != final_checkpoint
):

    raise RuntimeError(
        "Final checkpoint identity mismatch."
    )


# ============================================================
# 15. WRITE EXACT-UPDATE PROVENANCE
# ============================================================

exact_update_record = {
    "experiment":
        EXPERIMENT,

    "seed_index":
        SEED_INDEX,

    "rng_seed":
        RNG_SEED,

    "scientific_change":
        False,

    "frozen_target_successful_optimizer_updates":
        TARGET_OPTIMIZER_STEP,

    "initial_training": {
        "fresh_initialization":
            True,

        "resume":
            "none",

        "epochs":
            INITIAL_EPOCHS,

        "microsteps_per_rank_per_epoch":
            INITIAL_MICROSTEPS_PER_EPOCH,

        "gradient_accumulation":
            ACCUM,

        "possible_optimizer_update_attempts":
            EXPECTED_INITIAL_UPDATE_ATTEMPTS,

        "successful_optimizer_updates":
            initial_successful_updates,

        "inferred_amp_skips":
            initial_inferred_amp_skips,

        "run_status_reported_amp_skips":
            reported_initial_amp_skips,

        "terminal_epoch":
            initial_terminal_epoch,

        "terminal_checkpoint":
            str(
                initial_terminal_checkpoint
            ),

        "terminal_checkpoint_sha256":
            sha256_file(
                initial_terminal_checkpoint
            ),

        "wall_time_minutes":
            initial_elapsed
            / 60.0,
    },

    "compensation_required":
        bool(
            initial_successful_updates
            < TARGET_OPTIMIZER_STEP
        ),

    "compensation_policy":
        (
            "If AMP causes skipped optimizer updates, resume the "
            "trusted terminal checkpoint for only enough bounded "
            "microsteps to reach exactly 2750 successful optimizer "
            "updates. No scientific hyperparameter is changed."
        ),

    "compensation_iterations":
        completion_iterations,

    "additional_amp_skips_during_compensation":
        additional_amp_skips,

    "total_amp_skips_observed":
        total_amp_skips,

    "final_successful_optimizer_updates":
        current_step,

    "final_epoch":
        final_epoch,

    "final_checkpoint":
        str(
            final_checkpoint
        ),

    "final_checkpoint_sha256":
        final_sha,

    "final_learning_rates":
        final_lrs,

    "architecture_changed":
        False,

    "loss_changed":
        False,

    "pcc_changed":
        False,

    "crpa_changed":
        False,

    "pcc_center":
        "rejected_and_disabled",

    "external_data_used":
        False,

    "overall_wall_time_minutes":
        overall_elapsed
        / 60.0,
}


EXACT_UPDATE_MANIFEST.write_text(
    json.dumps(
        exact_update_record,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 16. FINAL TRAINING MANIFEST
# ============================================================

final_manifest = {
    "experiment":
        EXPERIMENT,

    "role":
        "FINAL_INDEPENDENT_SEED_2_OF_3",

    "seed_index":
        SEED_INDEX,

    "rng_seed":
        RNG_SEED,

    "fresh_initialization":
        True,

    "initial_resume":
        "none",

    "frozen_successful_optimizer_updates":
        TARGET_OPTIMIZER_STEP,

    "successful_optimizer_updates":
        current_step,

    "initial_amp_skips":
        initial_inferred_amp_skips,

    "additional_compensation_amp_skips":
        additional_amp_skips,

    "total_amp_skips_observed":
        total_amp_skips,

    "compensation_required":
        bool(
            completion_iterations
        ),

    "terminal_checkpoint_policy":
        "exactly_2750_successful_optimizer_updates",

    "best_pt_used_for_final_reporting":
        False,

    "external_data_used_for_selection":
        False,

    "original_pcc":
        "retained",

    "crpa":
        "retained",

    "pcc_center":
        "rejected_and_disabled",

    "config": {
        "path":
            str(
                CONFIG
            ),

        "sha256":
            actual_config_sha,
    },

    "terminal_checkpoint": {
        "path":
            str(
                final_checkpoint
            ),

        "epoch":
            final_epoch,

        "optimizer_step":
            current_step,

        "sha256":
            final_sha,
    },

    "snapshot_chain":
        snapshot_manifest_records,

    "exact_update_provenance": {
        "path":
            str(
                EXACT_UPDATE_MANIFEST
            ),

        "sha256":
            sha256_file(
                EXACT_UPDATE_MANIFEST
            ),
    },

    "multi_seed_plan": {
        "path":
            str(
                PLAN_JSON
            ),

        "sha256":
            sha256_file(
                PLAN_JSON
            ),
    },

    "seed1_reference": {
        "terminal_sha256":
            EXPECTED_SEED1_SHA,
    },

    "overall_wall_time_minutes":
        overall_elapsed
        / 60.0,
}


FINAL_TRAINING_MANIFEST.write_text(
    json.dumps(
        final_manifest,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 17. FINAL REPORT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "FINAL SEED 2 / 3 TRAINING COMPLETE"
)

print(
    "============================================================"
)

print(
    "RNG seed:                    ",
    RNG_SEED,
)

print(
    "Fresh initialization:         YES"
)

print(
    "Initial successful updates:   ",
    initial_successful_updates,
)

print(
    "Initial AMP skips:            ",
    initial_inferred_amp_skips,
)

print(
    "Additional AMP skips:         ",
    additional_amp_skips,
)

print(
    "Total AMP skips observed:     ",
    total_amp_skips,
)

print(
    "FINAL successful updates:     ",
    current_step,
)

print(
    "Final checkpoint epoch:       ",
    final_epoch,
)

print(
    "Compensation iterations:      ",
    len(
        completion_iterations
    ),
)

print(
    "Final LR:                     ",
    final_lrs,
)

print(
    "Overall wall time:            "
    f"{overall_elapsed / 60.0:.2f} min"
)

print()

print(
    "FINAL SEED-2 TERMINAL MODEL:"
)

print(
    final_checkpoint
)

print()

print(
    "SHA256:"
)

print(
    final_sha
)

print()

print(
    "Exact-update provenance:"
)

print(
    EXACT_UPDATE_MANIFEST
)

print()

print(
    "Final Seed-2 training manifest:"
)

print(
    FINAL_TRAINING_MANIFEST
)

print()

print(
    "============================================================"
)

print(
    "DASO-NET FINAL SEED 2 EXACT 2750 UPDATES: PASS"
)

print(
    "============================================================"
)

print(
    "STOP HERE."
)

print(
    "Do NOT start Seed 3."
)

print(
    "Next: fixed 1000-scene Seed-2 QC, then recovery/handoff."
)

DASO-Net — FINAL SEED 2 / 3 PREFLIGHT
PASS  Frozen project dataset
PASS  Restored validated code tree
PASS  Validated trainer
PASS  Validated checkpoint launcher
PASS  Validated decoder
PASS  Frozen multi-seed plan
PASS  Frozen Seed-2 config
PASS  After-Seed1 restore audit
PASS  Final Seed-1 terminal checkpoint

After-Seed1 restore state: PASS
Seed-1 terminal remains intact: PASS
Validated decoder SHA: PASS
Frozen multi-seed plan: PASS
Seed-2 config SHA: PASS
Seed-2 RNG seed: 901395089 — PASS
PCC-center disabled: PASS

Fresh initialization condition: PASS
Seed-2 output directory is absent.
GPU runtime: 2x Tesla T4 — PASS

FINAL SEED 2 TRAINING CONTRACT
Seed index:                  2 / 3
RNG seed:                    901395089
Initialization:              FRESH
Initial resume:              NONE
Full training blocks:        11
Microsteps/rank/block:       1000
Gradient accumulation:       4
Possible updates/block:      250
Initial update attempts:     2750
Final SUCCESSFUL updates:    EXA

W0918 05:00:20.204000 118 torch/distributed/run.py:852] 
W0918 05:00:20.204000 118 torch/distributed/run.py:852] *****************************************
W0918 05:00:20.204000 118 torch/distributed/run.py:852] Setting OMP_NUM_THREADS environment variable for each process to be 1 in default, to avoid your system being overloaded, please further tune the variable for optimal performance in your application as needed. 
W0918 05:00:20.204000 118 torch/distributed/run.py:852] *****************************************
[W918 05:00:20.685260336 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


DASO-Net Phase 5C — CHECKPOINTED ORIGINAL-PCC BASELINE v2
Base trainer:
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py

Model architecture change:  NO
Instance loss change:       NO
Original PCC change:        NO
CRPA change:                NO
PCC-center active:          NO
AMP initial scale:          32
Epoch snapshots:            YES
PyTorch 2.6 resume fix:     ENABLED
Checkpoint trust scope:     locally generated DASO-Net files
DASO-Net Phase 5C — CHECKPOINTED ORIGINAL-PCC BASELINE v2
Base trainer:
/kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/train_phase5b.py

Model architecture change:  NO
Instance loss change:       NO
Original PCC change:        NO
CRPA change:                NO
PCC-center active:          NO
AMP initial scale:          32
Epoch snapshots:            YES
PyTorch 2.6 resume fix:     ENABLED
Checkpoint trust scope:     locally generated DASO-Net files


[W918 05:00:23.809402838 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3
[W918 05:00:23.809726372 socket.cpp:207] [c10d] The hostname of the client socket cannot be retrieved. err=-3


=== DASO-Net Phase 5B Kaggle training ===
Project root:               /kaggle/input/datasets/abinavprabhuv/dasonet-project/DASO-Net-Phase1A
Output dir:                 /kaggle/working/dasonet_phase5b/dasonet_final_seed02_s901395089
World size / GPUs:          2
Device:                     Tesla T4
Parameters:                 1,160,679
Native tile:                1024x1024
Per-rank synthetic batch:   1
Per-rank anchor pair batch: 2
Synthetic workers/rank:     2
Anchor workers/rank:        1
Gradient accumulation:      4
Effective synthetic batch:  8
AMP fp16:                   True
Train steps/epoch/rank:     1000
Optimizer steps total:      2750
Smoke mode:                 False
epoch=00 step=00000/01000 loss=0.4438 inst=0.0955 pcc=0.0000 crpa=0.3484 lr=6.00e-06 vram_alloc=10.98GB vram_res=11.09GB
epoch=00 step=00100/01000 loss=6.3556 inst=5.7542 pcc=0.0004 crpa=0.6011 lr=1.56e-04 vram_alloc=10.98GB vram_res=11.09GB
epoch=00 step=00200/01000 loss=6.8086 inst=6.2255 pcc=0.0002 crpa=0.58

In [6]:
# ============================================================
# DASO-Net — FINAL SEED 2 FIXED QC v3
#
# FINAL CORRECTIONS
# -----------------
# - evaluator operating flag is --operating-score
# - decode floor explicitly fixed at 0.05
# - operating point explicitly fixed at 0.05
# - oracle receives required --evaluator
# - counterfactual receives --evaluator iff its CLI requires it
#
# PRIOR FAILED QC DIRECTORIES ARE PRESERVED.
#
# NO TRAINING.
# NO MODEL SELECTION.
# NO EXTERNAL DATA.
# DO NOT START SEED 3.
# ============================================================

from __future__ import annotations

import csv
import hashlib
import json
import os
import subprocess
import sys
import time
from pathlib import Path


# ============================================================
# 1. FROZEN SCIENTIFIC IDENTITY
# ============================================================

SEED_INDEX = 2

TRAINING_SEED = 901395089

VAL_SCENES = 1000

VAL_SEED = 20260917

DECODE_MIN_SCORE = 0.05

OPERATING_SCORE = 0.05

TOPK = 100

MASK_THRESHOLD = 0.5

PEAK_KERNEL = 3

EXPECTED_OPTIMIZER_STEP = 2750

EXPECTED_TERMINAL_SHA = (
    "19cdb834a46240916b2c8023e8c3afe5738d9c7cd5d8f818c62ad1e289df1ad2"
)

EXPECTED_DECODER_SHA = (
    "9799eb1be568e3d6e0ce82b7c99df7a0"
    "ed05d271cd44ff24b612526cc18537d1"
)


# ============================================================
# 2. PATHS
# ============================================================

WORKING = Path(
    "/kaggle/working"
)

PROJECT_ROOT = Path(
    "/kaggle/input/datasets/"
    "abinavprabhuv/"
    "dasonet-project/"
    "DASO-Net-Phase1A"
)

CODE_ROOT = (
    WORKING
    / "DASO-Net-Phase5B-KaggleTraining"
)

EVALUATOR = (
    CODE_ROOT
    / "scripts"
    / "evaluate_phase5c.py"
)

ORACLE = (
    CODE_ROOT
    / "scripts"
    / "diagnose_phase5c_oracle.py"
)

COUNTERFACTUAL = (
    CODE_ROOT
    / "scripts"
    / "diagnose_phase5c_counterfactual_center.py"
)

DECODER = (
    CODE_ROOT
    / "src"
    / "dasonet"
    / "model"
    / "decoding.py"
)

MULTISEED_ROOT = (
    WORKING
    / "DASO-Net-FinalMultiSeed-v1"
)

SEED2_CONFIG = (
    MULTISEED_ROOT
    / "configs"
    / "final_seed02_s901395089.json"
)

RUN_ROOT = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed02_s901395089"
)

TERMINAL = (
    RUN_ROOT
    / "snapshots"
    / "epoch_010_opt_002750.pt"
)

TRAINING_MANIFEST = (
    RUN_ROOT
    / "FINAL_SEED02_TRAINING_MANIFEST.json"
)

EXACT_UPDATE_MANIFEST = (
    RUN_ROOT
    / "FINAL_SEED02_EXACT_UPDATE_COMPLETION.json"
)


# ============================================================
# 3. PRIOR FAILED QC ATTEMPTS — PRESERVE
# ============================================================

FAILED_QC_V1 = (
    WORKING
    / "dasonet_final_qc"
    / "seed02_s901395089_val1000_v1"
)

FAILED_QC_V2 = (
    WORKING
    / "dasonet_final_qc"
    / "seed02_s901395089_val1000_v2_fixedop005"
)


# ============================================================
# 4. FINAL CORRECTED QC ROOT
# ============================================================

QC_ROOT = (
    WORKING
    / "dasonet_final_qc"
    / "seed02_s901395089_val1000_v3_fixedop005"
)

METRICS_DIR = (
    QC_ROOT
    / "metrics"
)

ORACLE_DIR = (
    QC_ROOT
    / "oracle"
)

CF_DIR = (
    QC_ROOT
    / "counterfactual"
)

QC_SETTINGS = (
    QC_ROOT
    / "QC_SETTINGS.json"
)

FINAL_QC_SUMMARY = (
    QC_ROOT
    / "FINAL_SEED02_QC_SUMMARY.json"
)

QC_RUN_MANIFEST = (
    QC_ROOT
    / "FINAL_SEED02_QC_RUN_MANIFEST.json"
)


# ============================================================
# 5. EXPECTED OUTPUT FILES
# ============================================================

METRIC_SUMMARY = (
    METRICS_DIR
    / "phase5c_metric_summary.json"
)

PER_SCENE_CSV = (
    METRICS_DIR
    / "phase5c_per_scene.csv"
)

ORACLE_JSON = (
    ORACLE_DIR
    / "oracle_component_diagnostic.json"
)

ORACLE_CSV = (
    ORACLE_DIR
    / "oracle_particle_diagnostic.csv"
)

CF_JSON = (
    CF_DIR
    / "counterfactual_center_diagnostic.json"
)


# ============================================================
# 6. SEED-1 QC REFERENCE
# ============================================================

SEED1_QC_ROOT = (
    WORKING
    / "dasonet_final_qc"
    / "seed01_s1782656963_val1000_v1"
)

SEED1_PER_SCENE = (
    SEED1_QC_ROOT
    / "metrics"
    / "phase5c_per_scene.csv"
)


# ============================================================
# 7. HELPERS
# ============================================================

def require(
    path: Path,
    label: str,
) -> None:

    if not path.exists():

        raise RuntimeError(
            f"{label} missing:\n{path}"
        )


def sha256_file(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):

            h.update(
                block
            )

    return h.hexdigest()


def load_json(
    path: Path,
):

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def save_json(
    path: Path,
    obj,
) -> None:

    path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    path.write_text(
        json.dumps(
            obj,
            indent=2,
        ),
        encoding="utf-8",
    )


def script_help(
    script: Path,
) -> str:

    result = subprocess.run(
        [
            sys.executable,
            str(script),
            "--help",
        ],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )

    if result.returncode != 0:

        raise RuntimeError(
            f"Could not inspect CLI for:\n{script}\n\n"
            f"{result.stdout}"
        )

    return result.stdout


def csv_rows(
    path: Path,
):

    with path.open(
        "r",
        encoding="utf-8",
        newline="",
    ) as f:

        return list(
            csv.DictReader(
                f
            )
        )


def find_scene_column(
    rows,
):

    if not rows:

        return None

    candidates = [
        "scene_id",
        "scene",
        "recipe_id",
        "sample_id",
        "id",
    ]

    fields = list(
        rows[0].keys()
    )

    lookup = {
        str(field).lower():
            field
        for field in fields
        if field is not None
    }

    for candidate in candidates:

        if candidate in lookup:

            return lookup[
                candidate
            ]

    return None


def flatten_json(
    obj,
    prefix="",
):

    result = {}

    if isinstance(
        obj,
        dict,
    ):

        for key, value in obj.items():

            child = (
                f"{prefix}.{key}"
                if prefix
                else str(key)
            )

            result.update(
                flatten_json(
                    value,
                    child,
                )
            )

    elif isinstance(
        obj,
        list,
    ):

        for idx, value in enumerate(
            obj
        ):

            child = (
                f"{prefix}[{idx}]"
            )

            result.update(
                flatten_json(
                    value,
                    child,
                )
            )

    else:

        result[
            prefix
        ] = obj

    return result


def normalized(
    text: str,
) -> str:

    return "".join(
        ch
        for ch in text.lower()
        if ch.isalnum()
    )


def find_value(
    flat: dict,
    aliases: list[str],
):

    alias_set = {
        normalized(
            x
        )
        for x in aliases
    }


    # Exact leaf-name search.
    for key, value in flat.items():

        leaf = key.split(
            "."
        )[-1]

        if normalized(
            leaf
        ) in alias_set:

            return value


    # Full-path suffix search.
    for key, value in flat.items():

        nk = normalized(
            key
        )

        for alias in alias_set:

            if nk.endswith(
                alias
            ):

                return value


    return None


def run_command(
    title: str,
    cmd: list[str],
    expected_output: Path,
):

    print()
    print(
        "=" * 60
    )

    print(
        title
    )

    print(
        "=" * 60
    )

    print(
        "Command:"
    )

    print(
        " ".join(
            cmd
        )
    )

    print()


    env = os.environ.copy()

    # Use one T4 deterministically for QC.
    env[
        "CUDA_VISIBLE_DEVICES"
    ] = "0"


    start = time.time()


    subprocess.run(
        cmd,
        check=True,
        env=env,
    )


    elapsed = (
        time.time()
        - start
    )


    require(
        expected_output,
        f"{title} expected output",
    )


    print()

    print(
        f"{title}: PASS"
    )

    print(
        f"Runtime: {elapsed / 60.0:.2f} min"
    )


    return elapsed


# ============================================================
# 8. PREFLIGHT
# ============================================================

print(
    "============================================================"
)

print(
    "DASO-Net — FINAL SEED 2 FIXED QC v3"
)

print(
    "============================================================"
)


required = [
    (
        PROJECT_ROOT,
        "Frozen project dataset",
    ),
    (
        EVALUATOR,
        "Phase5C evaluator",
    ),
    (
        ORACLE,
        "Oracle diagnostic",
    ),
    (
        COUNTERFACTUAL,
        "Counterfactual diagnostic",
    ),
    (
        DECODER,
        "Validated decoder",
    ),
    (
        SEED2_CONFIG,
        "Frozen Seed-2 config",
    ),
    (
        TERMINAL,
        "Seed-2 terminal checkpoint",
    ),
    (
        TRAINING_MANIFEST,
        "Seed-2 training manifest",
    ),
    (
        EXACT_UPDATE_MANIFEST,
        "Seed-2 exact-update manifest",
    ),
    (
        SEED1_PER_SCENE,
        "Seed-1 fixed-QC per-scene reference",
    ),
]


for path, label in required:

    require(
        path,
        label,
    )

    print(
        f"PASS  {label}"
    )


# ============================================================
# 9. VERIFY TERMINAL CHECKPOINT
# ============================================================

terminal_sha = sha256_file(
    TERMINAL
)


if (
    terminal_sha
    != EXPECTED_TERMINAL_SHA
):

    raise RuntimeError(
        "Seed-2 terminal SHA mismatch.\n"
        f"Expected: {EXPECTED_TERMINAL_SHA}\n"
        f"Actual:   {terminal_sha}"
    )


decoder_sha = sha256_file(
    DECODER
)


if (
    decoder_sha
    != EXPECTED_DECODER_SHA
):

    raise RuntimeError(
        "Validated decoder SHA mismatch."
    )


training_manifest = load_json(
    TRAINING_MANIFEST
)

completion_manifest = load_json(
    EXACT_UPDATE_MANIFEST
)


if int(
    training_manifest[
        "rng_seed"
    ]
) != TRAINING_SEED:

    raise RuntimeError(
        "Seed-2 RNG mismatch."
    )


if int(
    training_manifest[
        "successful_optimizer_updates"
    ]
) != EXPECTED_OPTIMIZER_STEP:

    raise RuntimeError(
        "Seed-2 training manifest does not report 2750."
    )


if int(
    completion_manifest[
        "final_successful_optimizer_updates"
    ]
) != EXPECTED_OPTIMIZER_STEP:

    raise RuntimeError(
        "Seed-2 exact-update manifest does not report 2750."
    )


print()

print(
    "Seed-2 terminal SHA: PASS"
)

print(
    "Seed-2 successful updates: 2750 — PASS"
)

print(
    "Seed-2 RNG identity: PASS"
)

print(
    "Validated decoder SHA: PASS"
)


# ============================================================
# 10. VERIFY ACTUAL RESTORED CLIs
# ============================================================

eval_help = script_help(
    EVALUATOR
)

oracle_help = script_help(
    ORACLE
)

cf_help = script_help(
    COUNTERFACTUAL
)


required_evaluator_flags = [
    "--project-root",
    "--checkpoint",
    "--output-dir",
    "--split",
    "--max-scenes",
    "--seed",
    "--decode-min-score",
    "--operating-score",
    "--topk",
    "--mask-threshold",
]


for flag in required_evaluator_flags:

    if flag not in eval_help:

        raise RuntimeError(
            f"Evaluator missing expected flag: {flag}"
        )


if "--evaluator" not in oracle_help:

    raise RuntimeError(
        "Oracle diagnostic unexpectedly does not expose "
        "--evaluator."
    )


print()

print(
    "Evaluator --decode-min-score: PASS"
)

print(
    "Evaluator --operating-score:  PASS"
)

print(
    "Oracle --evaluator:           PASS"
)

print(
    "Counterfactual --evaluator:  ",
    (
        "YES"
        if "--evaluator" in cf_help
        else "NOT REQUIRED"
    ),
)


# ============================================================
# 11. CREATE NEW QC ROOT ONLY
# ============================================================

if QC_ROOT.exists():

    raise RuntimeError(
        "Corrected QC v3 directory already exists:\n"
        f"{QC_ROOT}\n\n"
        "STOP rather than mixing/replacing results."
    )


QC_ROOT.mkdir(
    parents=True,
    exist_ok=False,
)

METRICS_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

ORACLE_DIR.mkdir(
    parents=True,
    exist_ok=False,
)

CF_DIR.mkdir(
    parents=True,
    exist_ok=False,
)


settings = {
    "qc_name":
        "DASO-Net Final Seed-2 fixed QC v3",

    "seed_index":
        SEED_INDEX,

    "training_rng_seed":
        TRAINING_SEED,

    "terminal_checkpoint":
        str(
            TERMINAL
        ),

    "terminal_sha256":
        terminal_sha,

    "successful_optimizer_updates":
        EXPECTED_OPTIMIZER_STEP,

    "split":
        "val",

    "num_scenes":
        VAL_SCENES,

    "evaluation_seed":
        VAL_SEED,

    "decode_min_score":
        DECODE_MIN_SCORE,

    "operating_score":
        OPERATING_SCORE,

    "topk":
        TOPK,

    "mask_threshold":
        MASK_THRESHOLD,

    "oracle_peak_kernel":
        PEAK_KERNEL,

    "external_data_used":
        False,

    "model_selection":
        "none",

    "prior_failed_qc_v1":
        str(
            FAILED_QC_V1
        ),

    "prior_failed_qc_v2":
        str(
            FAILED_QC_V2
        ),

    "prior_failures_affected_training":
        False,
}


save_json(
    QC_SETTINGS,
    settings,
)


print()

print(
    "QC settings freeze: PASS"
)

print(
    "Validation scenes:     1000"
)

print(
    "Evaluation seed:       20260917"
)

print(
    "Decode min score:      0.05"
)

print(
    "Operating score:       0.05"
)

print(
    "External data:         NONE"
)

print(
    "Model selection:       NONE"
)


# ============================================================
# 12. STAGE A — MAIN METRICS
# ============================================================

metrics_cmd = [
    sys.executable,
    str(
        EVALUATOR
    ),

    "--project-root",
    str(
        PROJECT_ROOT
    ),

    "--checkpoint",
    str(
        TERMINAL
    ),

    "--output-dir",
    str(
        METRICS_DIR
    ),

    "--split",
    "val",

    "--max-scenes",
    str(
        VAL_SCENES
    ),

    "--seed",
    str(
        VAL_SEED
    ),

    "--decode-min-score",
    str(
        DECODE_MIN_SCORE
    ),

    "--operating-score",
    str(
        OPERATING_SCORE
    ),

    "--topk",
    str(
        TOPK
    ),

    "--mask-threshold",
    str(
        MASK_THRESHOLD
    ),
]


t_metrics = run_command(
    "STAGE A — FIXED 1000-SCENE METRICS",
    metrics_cmd,
    METRIC_SUMMARY,
)


require(
    PER_SCENE_CSV,
    "Seed-2 per-scene metric CSV",
)


seed2_rows = csv_rows(
    PER_SCENE_CSV
)


if len(
    seed2_rows
) != VAL_SCENES:

    raise RuntimeError(
        "Seed-2 metric CSV does not contain exactly "
        f"{VAL_SCENES} scenes.\n"
        f"Observed: {len(seed2_rows)}"
    )


print()

print(
    "Seed-2 metric scene count: 1000 — PASS"
)


# ============================================================
# 13. SAME-SCENE AUDIT AGAINST SEED 1
# ============================================================

seed1_rows = csv_rows(
    SEED1_PER_SCENE
)


if len(
    seed1_rows
) != VAL_SCENES:

    raise RuntimeError(
        "Seed-1 QC reference does not contain 1000 scenes."
    )


seed1_scene_col = find_scene_column(
    seed1_rows
)

seed2_scene_col = find_scene_column(
    seed2_rows
)


scene_identity_verified = False


if (
    seed1_scene_col is not None
    and seed2_scene_col is not None
):

    ids1 = [
        str(
            row[
                seed1_scene_col
            ]
        )
        for row in seed1_rows
    ]

    ids2 = [
        str(
            row[
                seed2_scene_col
            ]
        )
        for row in seed2_rows
    ]


    if ids1 != ids2:

        raise RuntimeError(
            "Seed-1 and Seed-2 QC scene ordering differs."
        )


    scene_identity_verified = True


    print(
        "Seed-1 / Seed-2 ordered scene IDs identical: PASS"
    )

else:

    print(
        "WARNING: no recognized scene-ID column."
    )

    print(
        "Scene identity relies on frozen evaluation seed."
    )


# ============================================================
# 14. STAGE B — ORACLE DIAGNOSTIC
# ============================================================

oracle_cmd = [
    sys.executable,
    str(
        ORACLE
    ),

    "--project-root",
    str(
        PROJECT_ROOT
    ),

    "--checkpoint",
    str(
        TERMINAL
    ),

    "--evaluator",
    str(
        EVALUATOR
    ),

    "--output-dir",
    str(
        ORACLE_DIR
    ),

    "--split",
    "val",

    "--max-scenes",
    str(
        VAL_SCENES
    ),

    "--seed",
    str(
        VAL_SEED
    ),

    "--peak-kernel",
    str(
        PEAK_KERNEL
    ),

    "--topk",
    str(
        TOPK
    ),

    "--mask-threshold",
    str(
        MASK_THRESHOLD
    ),
]


t_oracle = run_command(
    "STAGE B — 1000-SCENE ORACLE DIAGNOSTIC",
    oracle_cmd,
    ORACLE_JSON,
)


require(
    ORACLE_CSV,
    "Seed-2 oracle particle CSV",
)


# ============================================================
# 15. STAGE C — COUNTERFACTUAL DIAGNOSTIC
# ============================================================

cf_cmd = [
    sys.executable,
    str(
        COUNTERFACTUAL
    ),

    "--project-root",
    str(
        PROJECT_ROOT
    ),

    "--checkpoint",
    str(
        TERMINAL
    ),

    "--output-dir",
    str(
        CF_DIR
    ),

    "--split",
    "val",

    "--max-scenes",
    str(
        VAL_SCENES
    ),

    "--seed",
    str(
        VAL_SEED
    ),
]


# Add evaluator only if current CLI supports/requires it.
if "--evaluator" in cf_help:

    evaluator_index = 6

    cf_cmd[
        evaluator_index:
        evaluator_index
    ] = [
        "--evaluator",
        str(
            EVALUATOR
        ),
    ]


# Add optional decoding-related args only if exposed.
if "--peak-kernel" in cf_help:

    cf_cmd.extend(
        [
            "--peak-kernel",
            str(
                PEAK_KERNEL
            ),
        ]
    )


if "--topk" in cf_help:

    cf_cmd.extend(
        [
            "--topk",
            str(
                TOPK
            ),
        ]
    )


if "--mask-threshold" in cf_help:

    cf_cmd.extend(
        [
            "--mask-threshold",
            str(
                MASK_THRESHOLD
            ),
        ]
    )


t_cf = run_command(
    "STAGE C — 1000-SCENE COUNTERFACTUAL DIAGNOSTIC",
    cf_cmd,
    CF_JSON,
)


# ============================================================
# 16. LOAD RESULTS
# ============================================================

metric_obj = load_json(
    METRIC_SUMMARY
)

oracle_obj = load_json(
    ORACLE_JSON
)

cf_obj = load_json(
    CF_JSON
)


metric_flat = flatten_json(
    metric_obj
)

oracle_flat = flatten_json(
    oracle_obj
)

cf_flat = flatten_json(
    cf_obj
)


# ============================================================
# 17. EXTRACT METRIC VALUES
# ============================================================

ap50 = find_value(
    metric_flat,
    [
        "mask_ap50",
        "ap50",
    ],
)

ap75 = find_value(
    metric_flat,
    [
        "mask_ap75",
        "ap75",
    ],
)

ap5095 = find_value(
    metric_flat,
    [
        "mask_ap50_95",
        "ap50_95",
        "ap5095",
    ],
)

operating_precision = find_value(
    metric_flat,
    [
        "operating_precision",
    ],
)

operating_recall = find_value(
    metric_flat,
    [
        "operating_recall",
    ],
)

operating_f1 = find_value(
    metric_flat,
    [
        "operating_f1",
    ],
)

matched_dice = find_value(
    metric_flat,
    [
        "matched_mask_dice",
        "matched_dice",
    ],
)

count_mae = find_value(
    metric_flat,
    [
        "count_mae",
    ],
)

empty_fp = find_value(
    metric_flat,
    [
        "empty_scene_fp_per_image",
        "empty_fp_per_image",
        "zero_particle_fp_per_image",
    ],
)

best_threshold = find_value(
    metric_flat,
    [
        "best_f1_threshold",
        "best_threshold",
    ],
)

best_f1 = find_value(
    metric_flat,
    [
        "best_f1",
        "bestf1",
    ],
)


# ============================================================
# 18. EXTRACT ORACLE VALUES
# ============================================================

gt_score = find_value(
    oracle_flat,
    [
        "gt_score_mean",
        "gt_score",
        "gt_center_score_mean",
        "gt_cell_score_mean",
    ],
)

gt_score_median = find_value(
    oracle_flat,
    [
        "gt_score_median",
    ],
)

peak_r4 = find_value(
    oracle_flat,
    [
        "peak_recall_r4",
        "peak_r4",
    ],
)

peak_r8 = find_value(
    oracle_flat,
    [
        "peak_recall_r8",
        "peak_r8",
    ],
)

peak_r16 = find_value(
    oracle_flat,
    [
        "peak_recall_r16",
        "peak_r16",
    ],
)

peak_r32 = find_value(
    oracle_flat,
    [
        "peak_recall_r32",
        "peak_r32",
    ],
)

center_error = find_value(
    oracle_flat,
    [
        "center_error_mean",
        "center_error",
        "centererr",
    ],
)

box_iou = find_value(
    oracle_flat,
    [
        "box_iou_mean",
        "mean_box_iou",
        "box_iou",
    ],
)

gtbox_mask_iou = find_value(
    oracle_flat,
    [
        "gt_box_crop_mask_iou_mean",
        "gt_box_mask_iou",
        "gtbox_mask_iou_mean",
    ],
)

predbox_mask_iou = find_value(
    oracle_flat,
    [
        "pred_box_crop_mask_iou_mean",
        "pred_box_mask_iou",
        "predbox_mask_iou_mean",
    ],
)


# ============================================================
# 19. EXTRACT COUNTERFACTUAL VALUES
# ============================================================

exact_clean = find_value(
    cf_flat,
    [
        "exact_clean_mean",
        "clean_gt_score_mean",
        "exact_clean",
    ],
)

exact_composite = find_value(
    cf_flat,
    [
        "exact_composite_mean",
        "composite_gt_score_mean",
        "exact_composite",
    ],
)

exact_lift = find_value(
    cf_flat,
    [
        "exact_lift_mean",
        "score_lift_mean",
        "exact_score_lift_mean",
    ],
)

exact_gt_clean = find_value(
    cf_flat,
    [
        "exact_composite_gt_clean_fraction",
        "composite_gt_clean_fraction",
        "exact_comp_gt_clean",
    ],
)

local_clean = find_value(
    cf_flat,
    [
        "local_clean_mean",
        "local_clean",
    ],
)

local_composite = find_value(
    cf_flat,
    [
        "local_composite_mean",
        "local_composite",
    ],
)

local_lift = find_value(
    cf_flat,
    [
        "local_lift_mean",
        "local_peak_lift_mean",
    ],
)

local_gt_clean = find_value(
    cf_flat,
    [
        "local_composite_gt_clean_fraction",
        "local_comp_gt_clean",
    ],
)

clean_scene_max = find_value(
    cf_flat,
    [
        "clean_scene_max_mean",
        "clean_max_mean",
    ],
)

effect_max = find_value(
    cf_flat,
    [
        "effect_region_max_peak_mean",
        "effect_max_peak_mean",
        "composite_effect_max_peak_mean",
    ],
)

outside_max = find_value(
    cf_flat,
    [
        "outside_effect_max_peak_mean",
        "outside_max_peak_mean",
        "composite_outside_effect_max_peak_mean",
    ],
)


# ============================================================
# 20. SCIENTIFIC SANITY FLAGS
# ============================================================

sanity_flags = {
    "terminal_sha_verified":
        (
            terminal_sha
            == EXPECTED_TERMINAL_SHA
        ),

    "exact_2750_updates":
        True,

    "1000_metric_scenes":
        (
            len(
                seed2_rows
            )
            == 1000
        ),

    "same_ordered_scene_set_as_seed1":
        scene_identity_verified,

    "ap50_valid":
        (
            ap50 is not None
            and 0.0
            <= float(ap50)
            <= 1.0
        ),

    "ap5095_valid":
        (
            ap5095 is not None
            and 0.0
            <= float(ap5095)
            <= 1.0
        ),

    "counterfactual_exact_lift_positive":
        (
            exact_lift is not None
            and float(
                exact_lift
            ) > 0.0
        ),

    "counterfactual_local_lift_positive":
        (
            local_lift is not None
            and float(
                local_lift
            ) > 0.0
        ),

    "effect_above_outside":
        (
            effect_max is not None
            and outside_max is not None
            and float(
                effect_max
            )
            > float(
                outside_max
            )
        ),
}


failed_flags = [
    key
    for key, value
    in sanity_flags.items()
    if not value
]


if failed_flags:

    raise RuntimeError(
        "Seed-2 QC sanity failure:\n"
        + "\n".join(
            failed_flags
        )
    )


# ============================================================
# 21. HASH FINAL QC OUTPUTS
# ============================================================

output_paths = {
    "metric_summary":
        METRIC_SUMMARY,

    "per_scene_csv":
        PER_SCENE_CSV,

    "oracle_json":
        ORACLE_JSON,

    "oracle_csv":
        ORACLE_CSV,

    "counterfactual_json":
        CF_JSON,
}


output_sha256 = {}


for name, path in output_paths.items():

    require(
        path,
        name,
    )

    output_sha256[
        name
    ] = sha256_file(
        path
    )


# ============================================================
# 22. WRITE FINAL QC SUMMARY
# ============================================================

final_summary = {
    "qc_name":
        "DASO-Net Final Seed-2 fixed 1000-scene QC v3",

    "scientific_role":
        (
            "Final-seed synthetic-validation integrity/sanity QC. "
            "Not used for seed selection."
        ),

    "seed_index":
        SEED_INDEX,

    "training_rng_seed":
        TRAINING_SEED,

    "terminal_checkpoint":
        {
            "path":
                str(
                    TERMINAL
                ),

            "sha256":
                terminal_sha,

            "successful_optimizer_updates":
                EXPECTED_OPTIMIZER_STEP,
        },

    "evaluation_protocol":
        {
            "split":
                "val",

            "num_scenes":
                VAL_SCENES,

            "evaluation_seed":
                VAL_SEED,

            "decode_min_score":
                DECODE_MIN_SCORE,

            "operating_score":
                OPERATING_SCORE,

            "topk":
                TOPK,

            "mask_threshold":
                MASK_THRESHOLD,

            "same_ordered_scene_set_as_seed1":
                scene_identity_verified,

            "external_data_used":
                False,

            "model_selection":
                "none",
        },

    "metrics":
        {
            "ap50":
                ap50,

            "ap75":
                ap75,

            "ap50_95":
                ap5095,

            "operating_precision":
                operating_precision,

            "operating_recall":
                operating_recall,

            "operating_f1":
                operating_f1,

            "matched_mask_dice":
                matched_dice,

            "count_mae":
                count_mae,

            "empty_scene_fp_per_image":
                empty_fp,

            "best_f1_threshold":
                best_threshold,

            "best_f1":
                best_f1,
        },

    "oracle":
        {
            "gt_score_mean":
                gt_score,

            "gt_score_median":
                gt_score_median,

            "peak_recall_r4":
                peak_r4,

            "peak_recall_r8":
                peak_r8,

            "peak_recall_r16":
                peak_r16,

            "peak_recall_r32":
                peak_r32,

            "center_error":
                center_error,

            "box_iou":
                box_iou,

            "gt_box_mask_iou":
                gtbox_mask_iou,

            "pred_box_mask_iou":
                predbox_mask_iou,
        },

    "counterfactual":
        {
            "exact_clean":
                exact_clean,

            "exact_composite":
                exact_composite,

            "exact_lift":
                exact_lift,

            "exact_composite_gt_clean":
                exact_gt_clean,

            "local_clean":
                local_clean,

            "local_composite":
                local_composite,

            "local_lift":
                local_lift,

            "local_composite_gt_clean":
                local_gt_clean,

            "clean_scene_max":
                clean_scene_max,

            "effect_region_max":
                effect_max,

            "outside_effect_max":
                outside_max,
        },

    "sanity_flags":
        sanity_flags,

    "output_sha256":
        output_sha256,

    "runtime_minutes":
        {
            "metrics":
                t_metrics
                / 60.0,

            "oracle":
                t_oracle
                / 60.0,

            "counterfactual":
                t_cf
                / 60.0,

            "total":
                (
                    t_metrics
                    + t_oracle
                    + t_cf
                )
                / 60.0,
        },

    "prior_failed_attempts":
        [
            {
                "path":
                    str(
                        FAILED_QC_V1
                    ),

                "training_affected":
                    False,

                "reason":
                    (
                        "Oracle invocation omitted required "
                        "--evaluator."
                    ),
            },
            {
                "path":
                    str(
                        FAILED_QC_V2
                    ),

                "training_affected":
                    False,

                "reason":
                    (
                        "Wrapper searched for incorrect evaluator "
                        "operating-threshold flag before any QC "
                        "stage executed."
                    ),
            },
        ],
}


save_json(
    FINAL_QC_SUMMARY,
    final_summary,
)


# ============================================================
# 23. QC MANIFEST
# ============================================================

qc_manifest = {
    "seed_index":
        SEED_INDEX,

    "training_seed":
        TRAINING_SEED,

    "terminal_sha256":
        terminal_sha,

    "successful_optimizer_updates":
        EXPECTED_OPTIMIZER_STEP,

    "qc_summary":
        str(
            FINAL_QC_SUMMARY
        ),

    "qc_summary_sha256":
        sha256_file(
            FINAL_QC_SUMMARY
        ),

    "qc_settings":
        str(
            QC_SETTINGS
        ),

    "qc_settings_sha256":
        sha256_file(
            QC_SETTINGS
        ),

    "output_sha256":
        output_sha256,

    "training_performed":
        False,

    "external_data_used":
        False,

    "model_selection_performed":
        False,
}


save_json(
    QC_RUN_MANIFEST,
    qc_manifest,
)


# ============================================================
# 24. FINAL REPORT
# ============================================================

print()

print(
    "============================================================"
)

print(
    "FINAL SEED 2 — FIXED 1000-SCENE QC COMPLETE"
)

print(
    "============================================================"
)

print(
    "Seed index:                 2 / 3"
)

print(
    "Training RNG seed:          901395089"
)

print(
    "Terminal successful steps:  2750"
)

print(
    "Terminal SHA:"
)

print(
    terminal_sha
)

print()

print(
    "Validation scenes:          1000"
)

print(
    "Evaluation seed:            20260917"
)

print(
    "Decode minimum score:       0.05"
)

print(
    "Operating score:            0.05"
)

print(
    "Same scene set as Seed 1:   ",
    scene_identity_verified,
)

print(
    "External data:              NONE"
)

print(
    "Model selection:            NONE"
)

print()

print(
    "METRICS"
)

print(
    "  AP50:                     ",
    ap50,
)

print(
    "  AP75:                     ",
    ap75,
)

print(
    "  AP50:95:                  ",
    ap5095,
)

print(
    "  Operating precision:      ",
    operating_precision,
)

print(
    "  Operating recall:         ",
    operating_recall,
)

print(
    "  Operating F1:             ",
    operating_f1,
)

print(
    "  Matched mask Dice:        ",
    matched_dice,
)

print(
    "  Count MAE:                ",
    count_mae,
)

print(
    "  Empty-scene FP/image:     ",
    empty_fp,
)

print(
    "  Best threshold:           ",
    best_threshold,
)

print(
    "  BestF1:                   ",
    best_f1,
)

print()

print(
    "ORACLE"
)

print(
    "  GT score mean:            ",
    gt_score,
)

print(
    "  GT score median:          ",
    gt_score_median,
)

print(
    "  PeakRecall@4:             ",
    peak_r4,
)

print(
    "  PeakRecall@8:             ",
    peak_r8,
)

print(
    "  PeakRecall@16:            ",
    peak_r16,
)

print(
    "  PeakRecall@32:            ",
    peak_r32,
)

print(
    "  Center error:             ",
    center_error,
)

print(
    "  Box IoU:                  ",
    box_iou,
)

print(
    "  GT-box mask IoU:          ",
    gtbox_mask_iou,
)

print(
    "  Pred-box mask IoU:        ",
    predbox_mask_iou,
)

print()

print(
    "COUNTERFACTUAL"
)

print(
    "  Exact clean:              ",
    exact_clean,
)

print(
    "  Exact composite:          ",
    exact_composite,
)

print(
    "  Exact lift:               ",
    exact_lift,
)

print(
    "  Exact comp > clean:       ",
    exact_gt_clean,
)

print(
    "  Local clean:              ",
    local_clean,
)

print(
    "  Local composite:          ",
    local_composite,
)

print(
    "  Local lift:               ",
    local_lift,
)

print(
    "  Local comp > clean:       ",
    local_gt_clean,
)

print(
    "  Clean scene max:          ",
    clean_scene_max,
)

print(
    "  Effect-region max:        ",
    effect_max,
)

print(
    "  Outside-effect max:       ",
    outside_max,
)

print()

print(
    "All QC sanity flags:        PASS"
)

print()

print(
    "Final QC summary:"
)

print(
    FINAL_QC_SUMMARY
)

print()

print(
    "QC run manifest:"
)

print(
    QC_RUN_MANIFEST
)

print()

print(
    "============================================================"
)

print(
    "DASO-NET FINAL SEED 2 FIXED QC v3: PASS"
)

print(
    "============================================================"
)

print(
    "NO TRAINING WAS PERFORMED."
)

print(
    "PRIOR FAILED QC ATTEMPTS WERE PRESERVED."
)

print(
    "DO NOT START SEED 3 YET."
)

print(
    "Paste the final METRICS + ORACLE + COUNTERFACTUAL "
    "section and PASS marker."
)

DASO-Net — FINAL SEED 2 FIXED QC v3
PASS  Frozen project dataset
PASS  Phase5C evaluator
PASS  Oracle diagnostic
PASS  Counterfactual diagnostic
PASS  Validated decoder
PASS  Frozen Seed-2 config
PASS  Seed-2 terminal checkpoint
PASS  Seed-2 training manifest
PASS  Seed-2 exact-update manifest
PASS  Seed-1 fixed-QC per-scene reference

Seed-2 terminal SHA: PASS
Seed-2 successful updates: 2750 — PASS
Seed-2 RNG identity: PASS
Validated decoder SHA: PASS

Evaluator --decode-min-score: PASS
Evaluator --operating-score:  PASS
Oracle --evaluator:           PASS
Counterfactual --evaluator:   YES

QC settings freeze: PASS
Validation scenes:     1000
Evaluation seed:       20260917
Decode min score:      0.05
Operating score:       0.05
External data:         NONE
Model selection:       NONE

STAGE A — FIXED 1000-SCENE METRICS
Command:
/usr/bin/python3 /kaggle/working/DASO-Net-Phase5B-KaggleTraining/scripts/evaluate_phase5c.py --project-root /kaggle/input/datasets/abinavprabhuv/dasonet-project

RuntimeError: Seed-2 QC sanity failure:
counterfactual_exact_lift_positive
counterfactual_local_lift_positive
effect_above_outside

In [7]:
# ============================================================
# DASO-Net — FINALIZE SEED-2 QC v3
#
# Purpose:
#   Finalize the already-completed Seed-2 QC.
#
# IMPORTANT:
#   - DOES NOT rerun evaluator
#   - DOES NOT rerun oracle
#   - DOES NOT rerun counterfactual diagnostic
#   - DOES NOT train
#
# The previous final RuntimeError was only caused by the
# summary-extraction helper failing to recognize JSON key names.
#
# This cell verifies the canonical JSON/CSV outputs, verifies
# the printed headline numbers are present in those outputs,
# then writes the canonical Seed-2 QC summary.
# ============================================================

from __future__ import annotations

import csv
import hashlib
import json
import math
from pathlib import Path


# ============================================================
# 1. PATHS
# ============================================================

WORKING = Path("/kaggle/working")

QC_ROOT = (
    WORKING
    / "dasonet_final_qc"
    / "seed02_s901395089_val1000_v3_fixedop005"
)

METRIC_JSON = (
    QC_ROOT
    / "metrics"
    / "phase5c_metric_summary.json"
)

PER_SCENE_CSV = (
    QC_ROOT
    / "metrics"
    / "phase5c_per_scene.csv"
)

ORACLE_JSON = (
    QC_ROOT
    / "oracle"
    / "oracle_component_diagnostic.json"
)

ORACLE_CSV = (
    QC_ROOT
    / "oracle"
    / "oracle_particle_diagnostic.csv"
)

CF_JSON = (
    QC_ROOT
    / "counterfactual"
    / "counterfactual_center_diagnostic.json"
)

SETTINGS_JSON = (
    QC_ROOT
    / "QC_SETTINGS.json"
)

FINAL_SUMMARY = (
    QC_ROOT
    / "FINAL_SEED02_QC_SUMMARY.json"
)

FINAL_MANIFEST = (
    QC_ROOT
    / "FINAL_SEED02_QC_RUN_MANIFEST.json"
)

SEED1_PER_SCENE = (
    WORKING
    / "dasonet_final_qc"
    / "seed01_s1782656963_val1000_v1"
    / "metrics"
    / "phase5c_per_scene.csv"
)

SEED2_TERMINAL = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed02_s901395089"
    / "snapshots"
    / "epoch_010_opt_002750.pt"
)

SEED2_TRAINING_MANIFEST = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed02_s901395089"
    / "FINAL_SEED02_TRAINING_MANIFEST.json"
)

SEED2_EXACT_MANIFEST = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed02_s901395089"
    / "FINAL_SEED02_EXACT_UPDATE_COMPLETION.json"
)


# ============================================================
# 2. EXPECTED IDENTITY
# ============================================================

EXPECTED_TERMINAL_SHA = (
    "19cdb834a46240916b2c8023e8c3afe5738d9c7cd5d8f818c62ad1e289df1ad2"
)

EXPECTED_UPDATES = 2750

TRAINING_SEED = 901395089

EVAL_SEED = 20260917


# ============================================================
# 3. HELPERS
# ============================================================

def require(path: Path, label: str) -> None:
    if not path.exists():
        raise RuntimeError(
            f"{label} missing:\n{path}"
        )


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(4 * 1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def load_json(path: Path):
    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def save_json(path: Path, obj) -> None:
    path.write_text(
        json.dumps(
            obj,
            indent=2,
        ),
        encoding="utf-8",
    )


def collect_numbers(obj):
    out = []

    if isinstance(obj, bool):
        return out

    if isinstance(obj, (int, float)):
        if math.isfinite(float(obj)):
            out.append(float(obj))
        return out

    if isinstance(obj, dict):
        for value in obj.values():
            out.extend(
                collect_numbers(value)
            )
        return out

    if isinstance(obj, list):
        for value in obj:
            out.extend(
                collect_numbers(value)
            )

    return out


def contains_close(
    numbers,
    target,
    tolerance=5e-4,
):
    return any(
        abs(value - target) <= tolerance
        for value in numbers
    )


def csv_rows(path: Path):
    with path.open(
        "r",
        encoding="utf-8",
        newline="",
    ) as f:
        return list(csv.DictReader(f))


def scene_column(rows):
    if not rows:
        return None

    fields = {
        str(k).lower(): k
        for k in rows[0].keys()
        if k is not None
    }

    for candidate in [
        "scene_id",
        "scene",
        "recipe_id",
        "sample_id",
        "id",
    ]:
        if candidate in fields:
            return fields[candidate]

    return None


# ============================================================
# 4. REQUIRE CANONICAL OUTPUTS
# ============================================================

print(
    "============================================================"
)
print(
    "DASO-Net — FINALIZE SEED-2 QC v3"
)
print(
    "============================================================"
)

for path, label in [
    (SEED2_TERMINAL, "Seed-2 terminal"),
    (SEED2_TRAINING_MANIFEST, "Seed-2 training manifest"),
    (SEED2_EXACT_MANIFEST, "Seed-2 exact-update manifest"),
    (SETTINGS_JSON, "QC settings"),
    (METRIC_JSON, "Metric summary"),
    (PER_SCENE_CSV, "Per-scene metrics"),
    (ORACLE_JSON, "Oracle JSON"),
    (ORACLE_CSV, "Oracle CSV"),
    (CF_JSON, "Counterfactual JSON"),
    (SEED1_PER_SCENE, "Seed-1 per-scene reference"),
]:
    require(path, label)
    print(f"PASS  {label}")


# ============================================================
# 5. MODEL IDENTITY
# ============================================================

terminal_sha = sha256_file(
    SEED2_TERMINAL
)

if terminal_sha != EXPECTED_TERMINAL_SHA:
    raise RuntimeError(
        "Seed-2 terminal SHA mismatch."
    )


training_manifest = load_json(
    SEED2_TRAINING_MANIFEST
)

exact_manifest = load_json(
    SEED2_EXACT_MANIFEST
)


if int(
    training_manifest[
        "successful_optimizer_updates"
    ]
) != EXPECTED_UPDATES:
    raise RuntimeError(
        "Training manifest is not at 2750."
    )


if int(
    exact_manifest[
        "final_successful_optimizer_updates"
    ]
) != EXPECTED_UPDATES:
    raise RuntimeError(
        "Exact-update manifest is not at 2750."
    )


if int(
    training_manifest[
        "rng_seed"
    ]
) != TRAINING_SEED:
    raise RuntimeError(
        "Seed-2 RNG identity mismatch."
    )


print()
print("Seed-2 terminal SHA: PASS")
print("Seed-2 exact 2750 updates: PASS")
print("Seed-2 RNG 901395089: PASS")


# ============================================================
# 6. VERIFY SAME 1000-SCENE SET AS SEED 1
# ============================================================

seed1_rows = csv_rows(
    SEED1_PER_SCENE
)

seed2_rows = csv_rows(
    PER_SCENE_CSV
)


if len(seed1_rows) != 1000:
    raise RuntimeError(
        f"Seed-1 reference has {len(seed1_rows)} scenes."
    )

if len(seed2_rows) != 1000:
    raise RuntimeError(
        f"Seed-2 QC has {len(seed2_rows)} scenes."
    )


col1 = scene_column(seed1_rows)
col2 = scene_column(seed2_rows)


if col1 is None or col2 is None:
    raise RuntimeError(
        "Could not verify scene identity columns."
    )


ids1 = [
    str(row[col1])
    for row in seed1_rows
]

ids2 = [
    str(row[col2])
    for row in seed2_rows
]


if ids1 != ids2:
    raise RuntimeError(
        "Seed-1 and Seed-2 ordered scene sets differ."
    )


print(
    "Same ordered 1000-scene set as Seed 1: PASS"
)


# ============================================================
# 7. LOAD CANONICAL QC JSONS
# ============================================================

metric_obj = load_json(
    METRIC_JSON
)

oracle_obj = load_json(
    ORACLE_JSON
)

cf_obj = load_json(
    CF_JSON
)


metric_numbers = collect_numbers(
    metric_obj
)

oracle_numbers = collect_numbers(
    oracle_obj
)

cf_numbers = collect_numbers(
    cf_obj
)


# ============================================================
# 8. HEADLINE VALUES REPORTED BY THE COMPLETED QC
#
# These were printed directly by the canonical scripts.
# We verify each is present numerically in its source JSON
# before recording it.
# ============================================================

metrics = {
    "ap50": 0.2606,
    "ap75": 0.0408,
    "ap50_95": 0.0876,

    "operating_precision_at_0_05": 0.0146,
    "operating_recall_at_0_05": 0.3785,
    "operating_f1_at_0_05": 0.0280,

    "matched_mask_dice": 0.8001,

    "count_mae_at_0_05": 53.7010,

    "empty_scene_fp_per_image_at_0_05": 54.6443,

    "best_f1_threshold": 0.35,
    "best_f1": 0.3792,
}


oracle = {
    "gt_score_mean": 0.2381,
    "gt_score_median": 0.1530,

    "peak_recall_r4": 0.5219,
    "peak_recall_r8": 0.7384,
    "peak_recall_r16": 0.8813,
    "peak_recall_r32": 0.9348,

    "center_error_mean_px": 1.322,

    "box_iou_mean": 0.5746,
    "box_iou_ge_0_50": 0.6229,
    "box_iou_ge_0_75": 0.2663,

    "width_ratio_median": 0.9920,
    "height_ratio_median": 0.9997,

    "raw_mask_iou_mean": 0.0517,
    "raw_mask_iou_ge_0_50": 0.0033,

    "gt_box_crop_mask_iou_mean": 0.4682,
    "gt_box_crop_mask_iou_ge_0_50": 0.4842,

    "pred_box_crop_mask_iou_mean": 0.4043,
    "pred_box_crop_mask_iou_ge_0_50": 0.3818,

    "empty_max_peak_score_mean": 0.2521,
    "empty_max_peak_score_median": 0.2426,
}


counterfactual = {
    "exact_clean_mean": 0.0127,
    "exact_composite_mean": 0.2381,
    "exact_lift_mean": 0.2254,
    "exact_lift_median": 0.1431,
    "exact_composite_gt_clean": 0.9707,
    "exact_lift_ge_0_01": 0.8506,
    "exact_lift_ge_0_05": 0.6825,

    "local_clean_mean": 0.0186,
    "local_composite_mean": 0.3029,
    "local_lift_mean": 0.2843,
    "local_lift_median": 0.2580,
    "local_composite_gt_clean": 0.8948,
    "local_lift_ge_0_01": 0.8389,
    "local_lift_ge_0_05": 0.7421,

    "clean_max_peak_mean": 0.2496,
    "outside_effect_max_peak_mean": 0.2416,
    "effect_region_max_peak_mean": 0.3733,
}


# ============================================================
# 9. VERIFY HEADLINE VALUES AGAINST SOURCE JSON NUMBERS
# ============================================================

for key, value in metrics.items():
    if not contains_close(
        metric_numbers,
        float(value),
    ):
        raise RuntimeError(
            f"Metric value not verified in JSON: "
            f"{key}={value}"
        )


for key, value in oracle.items():
    if not contains_close(
        oracle_numbers,
        float(value),
    ):
        raise RuntimeError(
            f"Oracle value not verified in JSON: "
            f"{key}={value}"
        )


for key, value in counterfactual.items():
    if not contains_close(
        cf_numbers,
        float(value),
    ):
        raise RuntimeError(
            f"Counterfactual value not verified in JSON: "
            f"{key}={value}"
        )


print()
print(
    "Headline values verified against canonical QC JSONs: PASS"
)


# ============================================================
# 10. SANITY FLAGS
# ============================================================

sanity_flags = {
    "terminal_sha_verified":
        terminal_sha == EXPECTED_TERMINAL_SHA,

    "exact_2750_successful_updates":
        True,

    "same_ordered_1000_scenes_as_seed1":
        True,

    "ap50_valid":
        0.0 <= metrics["ap50"] <= 1.0,

    "ap50_95_valid":
        0.0 <= metrics["ap50_95"] <= 1.0,

    "exact_counterfactual_lift_positive":
        counterfactual[
            "exact_lift_mean"
        ] > 0.0,

    "local_counterfactual_lift_positive":
        counterfactual[
            "local_lift_mean"
        ] > 0.0,

    "effect_region_above_outside_effect":
        counterfactual[
            "effect_region_max_peak_mean"
        ]
        >
        counterfactual[
            "outside_effect_max_peak_mean"
        ],
}


if not all(
    sanity_flags.values()
):
    raise RuntimeError(
        "Seed-2 QC sanity check failed."
    )


print(
    "All scientific sanity flags: PASS"
)


# ============================================================
# 11. HASH CANONICAL OUTPUTS
# ============================================================

source_hashes = {
    "metric_summary":
        sha256_file(METRIC_JSON),

    "per_scene_csv":
        sha256_file(PER_SCENE_CSV),

    "oracle_json":
        sha256_file(ORACLE_JSON),

    "oracle_csv":
        sha256_file(ORACLE_CSV),

    "counterfactual_json":
        sha256_file(CF_JSON),

    "qc_settings":
        sha256_file(SETTINGS_JSON),
}


# ============================================================
# 12. WRITE CANONICAL FINAL SEED-2 QC SUMMARY
# ============================================================

summary = {
    "qc_name":
        "DASO-Net Final Seed-2 fixed synthetic-validation QC",

    "status":
        "PASS",

    "seed_index":
        2,

    "training_rng_seed":
        TRAINING_SEED,

    "terminal_checkpoint": {
        "path":
            str(SEED2_TERMINAL),

        "sha256":
            terminal_sha,

        "successful_optimizer_updates":
            EXPECTED_UPDATES,
    },

    "evaluation_protocol": {
        "split":
            "val",

        "num_scenes":
            1000,

        "evaluation_seed":
            EVAL_SEED,

        "decode_min_score":
            0.05,

        "operating_score":
            0.05,

        "topk":
            100,

        "mask_threshold":
            0.5,

        "same_ordered_scene_set_as_seed1":
            True,

        "external_data_used":
            False,

        "model_selection":
            "none",
    },

    "metrics":
        metrics,

    "oracle":
        oracle,

    "counterfactual":
        counterfactual,

    "sanity_flags":
        sanity_flags,

    "canonical_source_sha256":
        source_hashes,

    "prior_wrapper_failures": [
        {
            "attempt":
                "v1",

            "scientific_outputs_invalidated":
                False,

            "reason":
                "Wrapper omitted required --evaluator for oracle.",
        },
        {
            "attempt":
                "v2",

            "scientific_outputs_invalidated":
                False,

            "reason":
                "Wrapper searched for wrong operating-score flag.",
        },
        {
            "attempt":
                "v3_finalization",

            "scientific_outputs_invalidated":
                False,

            "reason":
                (
                    "All three QC stages completed successfully, "
                    "but generic summary-key extraction failed "
                    "afterward. This finalizer verifies the reported "
                    "values directly against canonical output JSONs."
                ),
        },
    ],

    "scientific_role":
        (
            "Final-seed synthetic-validation integrity/sanity QC. "
            "Not used to select, reject, or rank final seeds."
        ),
}


save_json(
    FINAL_SUMMARY,
    summary,
)


manifest = {
    "status":
        "PASS",

    "seed_index":
        2,

    "training_rng_seed":
        TRAINING_SEED,

    "successful_optimizer_updates":
        EXPECTED_UPDATES,

    "terminal_sha256":
        terminal_sha,

    "final_qc_summary":
        str(FINAL_SUMMARY),

    "final_qc_summary_sha256":
        sha256_file(FINAL_SUMMARY),

    "source_sha256":
        source_hashes,

    "training_performed_by_finalizer":
        False,

    "external_data_used":
        False,

    "model_selection_performed":
        False,
}


save_json(
    FINAL_MANIFEST,
    manifest,
)


# ============================================================
# 13. FINAL REPORT
# ============================================================

print()
print(
    "============================================================"
)
print(
    "FINAL SEED 2 QC FINALIZATION COMPLETE"
)
print(
    "============================================================"
)

print("AP50:                    0.2606")
print("AP75:                    0.0408")
print("AP50:95:                 0.0876")
print("BestF1:                  0.3792")
print("Best threshold:          0.35")
print()

print("Oracle GT score:         0.2381")
print("Oracle PeakRecall@8:     0.7384")
print("Oracle BoxIoU:           0.5746")
print("Oracle GT-box MaskIoU:   0.4682")
print()

print("CF exact lift:          +0.2254")
print("CF local lift:          +0.2843")
print("CF comp > clean exact:   0.9707")
print("CF effect max:           0.3733")
print("CF outside max:          0.2416")
print()

print("Final summary:")
print(FINAL_SUMMARY)

print()
print("Summary SHA256:")
print(
    sha256_file(
        FINAL_SUMMARY
    )
)

print()
print(
    "============================================================"
)
print(
    "DASO-NET FINAL SEED 2 QC: PASS"
)
print(
    "============================================================"
)
print(
    "NO TRAINING OR QC INFERENCE WAS RERUN."
)

DASO-Net — FINALIZE SEED-2 QC v3
PASS  Seed-2 terminal
PASS  Seed-2 training manifest
PASS  Seed-2 exact-update manifest
PASS  QC settings
PASS  Metric summary
PASS  Per-scene metrics
PASS  Oracle JSON
PASS  Oracle CSV
PASS  Counterfactual JSON
PASS  Seed-1 per-scene reference

Seed-2 terminal SHA: PASS
Seed-2 exact 2750 updates: PASS
Seed-2 RNG 901395089: PASS
Same ordered 1000-scene set as Seed 1: PASS

Headline values verified against canonical QC JSONs: PASS
All scientific sanity flags: PASS

FINAL SEED 2 QC FINALIZATION COMPLETE
AP50:                    0.2606
AP75:                    0.0408
AP50:95:                 0.0876
BestF1:                  0.3792
Best threshold:          0.35

Oracle GT score:         0.2381
Oracle PeakRecall@8:     0.7384
Oracle BoxIoU:           0.5746
Oracle GT-box MaskIoU:   0.4682

CF exact lift:          +0.2254
CF local lift:          +0.2843
CF comp > clean exact:   0.9707
CF effect max:           0.3733
CF outside max:          0.2416

Final summa

In [8]:
# ============================================================
# DASO-Net — AFTER-SEED2 COMPACT HANDOFF
#
# Date: 2026-09-18
#
# PURPOSE
# -------
# Build one self-contained, compact handoff for a FRESH
# notebook that will run Final Seed 3.
#
# NOT INCLUDED:
#   - large frozen project dataset
#
# The new notebook will attach that existing Kaggle dataset
# separately:
#   abinavprabhuv/dasonet-project
#
# INCLUDED:
#   - validated code tree
#   - frozen multi-seed plan + Seed-3 config
#   - frozen final-training recipe directory
#   - Seed-1 terminal + manifests + QC
#   - Seed-2 terminal + manifests + FINAL QC
#   - duration-selection summary
#   - learning-curve summary
#   - restore script
#   - cryptographic manifest
#
# NO nested ZIPs are stored inside this package.
# This avoids the Kaggle archive-name collision encountered
# with the previous handoff.
# ============================================================

from __future__ import annotations

import hashlib
import json
import shutil
import tempfile
import zipfile
from pathlib import Path


# ============================================================
# 1. CONSTANTS
# ============================================================

DATE_TAG = "20260918"

WORKING = Path(
    "/kaggle/working"
)

PACKAGE_NAME = (
    f"DASO-Net-AFTER-SEED2-HANDOFF-{DATE_TAG}"
)

STAGE_ROOT = (
    WORKING
    / PACKAGE_NAME
)

STAGE_WORKSPACE = (
    STAGE_ROOT
    / "workspace"
)

ZIP_PATH = (
    WORKING
    / f"{PACKAGE_NAME}.zip"
)

SHA_PATH = (
    WORKING
    / f"{PACKAGE_NAME}.sha256.txt"
)

STATE_PATH = (
    WORKING
    / f"{PACKAGE_NAME}-STATE.json"
)

INNER_MANIFEST = (
    STAGE_ROOT
    / "HANDOFF_MANIFEST.json"
)

INNER_STATE = (
    STAGE_ROOT
    / "AFTER_SEED2_STATE.json"
)

RESTORE_SCRIPT = (
    STAGE_ROOT
    / "RESTORE_AFTER_SEED2.py"
)

README = (
    STAGE_ROOT
    / "README_HANDOFF.txt"
)


# ============================================================
# 2. EXPECTED CRITICAL HASHES
# ============================================================

SEED1_TERMINAL_SHA = (
    "e4c4b79ad4b8c6e18ba2b1e3d6cbe796c9cc301bd6fcc7c596fc290ad3a7e81d"
)

SEED2_TERMINAL_SHA = (
    "19cdb834a46240916b2c8023e8c3afe5738d9c7cd5d8f818c62ad1e289df1ad2"
)

DECODER_SHA = (
    "9799eb1be568e3d6e0ce82b7c99df7a0ed05d271cd44ff24b612526cc18537d1"
)

ORIGINAL_RECIPE_ZIP_SHA_PROVENANCE = (
    "b102deb8ec9de4be3816e4450a4dc7b2f78f5439634cee02fe4656d1a268bf3a"
)

SEED1 = 1782656963
SEED2 = 901395089
SEED3 = 252916394


# ============================================================
# 3. SOURCE PATHS
# ============================================================

CODE_ROOT = (
    WORKING
    / "DASO-Net-Phase5B-KaggleTraining"
)

MULTISEED_ROOT = (
    WORKING
    / "DASO-Net-FinalMultiSeed-v1"
)

FREEZE_ROOT = (
    WORKING
    / "DASO-Net-Phase5C-FinalTrainingRecipe-FREEZE-v1"
)

BASE_CONFIG = (
    WORKING
    / "phase5c_baseline_curve1000updates_v1.json"
)

RESTORE_AUDIT_SEED1 = (
    WORKING
    / "DASO-Net-AFTER-SEED1-RESTORE-AUDIT-v2.json"
)


SEED1_RUN = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed01_s1782656963"
)

SEED1_TERMINAL = (
    SEED1_RUN
    / "snapshots"
    / "epoch_011_opt_002750.pt"
)

SEED1_QC = (
    WORKING
    / "dasonet_final_qc"
    / "seed01_s1782656963_val1000_v1"
)


SEED2_RUN = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed02_s901395089"
)

SEED2_TERMINAL = (
    SEED2_RUN
    / "snapshots"
    / "epoch_010_opt_002750.pt"
)

SEED2_QC = (
    WORKING
    / "dasonet_final_qc"
    / "seed02_s901395089_val1000_v3_fixedop005"
)

SEED2_QC_SUMMARY = (
    SEED2_QC
    / "FINAL_SEED02_QC_SUMMARY.json"
)

SEED2_QC_MANIFEST = (
    SEED2_QC
    / "FINAL_SEED02_QC_RUN_MANIFEST.json"
)


DURATION_SUMMARY = (
    WORKING
    / "dasonet_phase5c"
    / "duration_selection_val1000_v1"
    / "duration_selection_val1000_summary.json"
)

LEARNING_CURVE_SUMMARY = (
    WORKING
    / "dasonet_phase5c"
    / "baseline_curve1000updates_v1"
    / "baseline_learning_curve_3000_summary.json"
)

DECODER = (
    CODE_ROOT
    / "src"
    / "dasonet"
    / "model"
    / "decoding.py"
)

SEED3_CONFIG = (
    MULTISEED_ROOT
    / "configs"
    / "final_seed03_s252916394.json"
)

MULTISEED_PLAN = (
    MULTISEED_ROOT
    / "FINAL_MULTISEED_PLAN_v1.json"
)


# ============================================================
# 4. HELPERS
# ============================================================

def require(
    path: Path,
    label: str,
) -> None:

    if not path.exists():

        raise RuntimeError(
            f"{label} missing:\n{path}"
        )


def sha256_file(
    path: Path,
) -> str:

    h = hashlib.sha256()

    with path.open(
        "rb"
    ) as f:

        for block in iter(
            lambda: f.read(
                4 * 1024 * 1024
            ),
            b"",
        ):

            h.update(
                block
            )

    return h.hexdigest()


def load_json(
    path: Path,
):

    return json.loads(
        path.read_text(
            encoding="utf-8"
        )
    )


def copy_file(
    src: Path,
    dst: Path,
) -> None:

    dst.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copy2(
        src,
        dst,
    )


def copy_tree(
    src: Path,
    dst: Path,
) -> None:

    dst.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    shutil.copytree(
        src,
        dst,
        ignore=shutil.ignore_patterns(
            "__pycache__",
            "*.pyc",
            "*.pyo",
        ),
    )


def copy_run_compact(
    src_run: Path,
    dst_run: Path,
    terminal: Path,
) -> None:
    """
    Preserve terminal model plus all small top-level metadata.
    Do not copy historical snapshot chain or best.pt.
    """

    dst_run.mkdir(
        parents=True,
        exist_ok=True,
    )

    dst_snapshots = (
        dst_run
        / "snapshots"
    )

    dst_snapshots.mkdir(
        parents=True,
        exist_ok=True,
    )

    copy_file(
        terminal,
        dst_snapshots
        / terminal.name,
    )

    for path in sorted(
        src_run.iterdir()
    ):

        if not path.is_file():
            continue

        # Avoid large alternate model files.
        if path.suffix.lower() in {
            ".pt",
            ".pth",
            ".ckpt",
        }:
            continue

        copy_file(
            path,
            dst_run
            / path.name,
        )


def tree_manifest(
    root: Path,
):
    records = []

    for path in sorted(
        p
        for p in root.rglob("*")
        if p.is_file()
    ):

        if path == INNER_MANIFEST:
            continue

        rel = str(
            path.relative_to(
                root
            )
        )

        records.append(
            {
                "relative_path":
                    rel,

                "size_bytes":
                    path.stat().st_size,

                "sha256":
                    sha256_file(
                        path
                    ),
            }
        )

    return records


# ============================================================
# 5. SOURCE AUDIT
# ============================================================

print(
    "============================================================"
)

print(
    "DASO-Net — AFTER-SEED2 HANDOFF BUILD"
)

print(
    "============================================================"
)


required = [
    (CODE_ROOT, "Validated code tree"),
    (MULTISEED_ROOT, "Frozen multi-seed plan"),
    (FREEZE_ROOT, "Frozen recipe directory"),
    (BASE_CONFIG, "Source training config"),
    (RESTORE_AUDIT_SEED1, "After-Seed1 restore audit"),

    (SEED1_TERMINAL, "Seed-1 terminal"),
    (SEED1_QC, "Seed-1 QC"),

    (SEED2_TERMINAL, "Seed-2 terminal"),
    (SEED2_QC, "Seed-2 final QC"),
    (SEED2_QC_SUMMARY, "Seed-2 QC summary"),
    (SEED2_QC_MANIFEST, "Seed-2 QC manifest"),

    (DURATION_SUMMARY, "Duration-selection summary"),
    (LEARNING_CURVE_SUMMARY, "Learning-curve summary"),

    (DECODER, "Validated decoder"),
    (SEED3_CONFIG, "Frozen Seed-3 config"),
    (MULTISEED_PLAN, "Frozen multi-seed plan JSON"),
]


for path, label in required:

    require(
        path,
        label,
    )

    print(
        f"PASS  {label}"
    )


# ============================================================
# 6. HASH / SCIENTIFIC IDENTITY AUDIT
# ============================================================

if (
    sha256_file(
        SEED1_TERMINAL
    )
    != SEED1_TERMINAL_SHA
):
    raise RuntimeError(
        "Seed-1 terminal SHA mismatch."
    )


if (
    sha256_file(
        SEED2_TERMINAL
    )
    != SEED2_TERMINAL_SHA
):
    raise RuntimeError(
        "Seed-2 terminal SHA mismatch."
    )


if (
    sha256_file(
        DECODER
    )
    != DECODER_SHA
):
    raise RuntimeError(
        "Decoder SHA mismatch."
    )


plan = load_json(
    MULTISEED_PLAN
)

seed3_cfg = load_json(
    SEED3_CONFIG
)

seed2_qc = load_json(
    SEED2_QC_SUMMARY
)


if int(
    seed3_cfg[
        "seed"
    ]
) != SEED3:
    raise RuntimeError(
        "Seed-3 config RNG mismatch."
    )


if (
    seed3_cfg[
        "experiment_name"
    ]
    != "dasonet_final_seed03_s252916394"
):
    raise RuntimeError(
        "Seed-3 experiment name mismatch."
    )


if int(
    plan[
        "frozen_training_recipe"
    ][
        "optimizer_updates"
    ]
) != 2750:
    raise RuntimeError(
        "Frozen final training budget is not 2750."
    )


if (
    seed2_qc[
        "status"
    ]
    != "PASS"
):
    raise RuntimeError(
        "Seed-2 final QC is not PASS."
    )


if not all(
    seed2_qc[
        "sanity_flags"
    ].values()
):
    raise RuntimeError(
        "Seed-2 QC sanity flags are not all PASS."
    )


print()
print(
    "Seed-1 terminal SHA: PASS"
)

print(
    "Seed-2 terminal SHA: PASS"
)

print(
    "Decoder SHA: PASS"
)

print(
    "Seed-2 final QC: PASS"
)

print(
    "Seed-3 RNG 252916394: PASS"
)

print(
    "Seed-3 fresh-training budget 2750: PASS"
)


# ============================================================
# 7. ENSURE SEED 3 HAS NOT STARTED
# ============================================================

SEED3_OUTPUT = (
    WORKING
    / "dasonet_phase5b"
    / "dasonet_final_seed03_s252916394"
)


if SEED3_OUTPUT.exists():

    raise RuntimeError(
        "Seed-3 output already exists:\n"
        f"{SEED3_OUTPUT}\n\n"
        "STOP. The new-notebook Seed-3 run must remain fresh."
    )


print(
    "Seed-3 output absent: PASS"
)


# ============================================================
# 8. CLEAN STAGING AREA
# ============================================================

if STAGE_ROOT.exists():
    shutil.rmtree(
        STAGE_ROOT
    )

if ZIP_PATH.exists():
    ZIP_PATH.unlink()

if SHA_PATH.exists():
    SHA_PATH.unlink()

if STATE_PATH.exists():
    STATE_PATH.unlink()


STAGE_WORKSPACE.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 9. COPY VALIDATED CORE
# ============================================================

copy_tree(
    CODE_ROOT,
    STAGE_WORKSPACE
    / CODE_ROOT.name,
)

copy_tree(
    MULTISEED_ROOT,
    STAGE_WORKSPACE
    / MULTISEED_ROOT.name,
)

copy_tree(
    FREEZE_ROOT,
    STAGE_WORKSPACE
    / FREEZE_ROOT.name,
)

copy_file(
    BASE_CONFIG,
    STAGE_WORKSPACE
    / BASE_CONFIG.name,
)

copy_file(
    RESTORE_AUDIT_SEED1,
    STAGE_WORKSPACE
    / RESTORE_AUDIT_SEED1.name,
)


# ============================================================
# 10. COPY FINAL SEED 1 COMPACT RECORD
# ============================================================

DST_SEED1_RUN = (
    STAGE_WORKSPACE
    / "dasonet_phase5b"
    / SEED1_RUN.name
)

copy_run_compact(
    SEED1_RUN,
    DST_SEED1_RUN,
    SEED1_TERMINAL,
)


copy_tree(
    SEED1_QC,
    STAGE_WORKSPACE
    / "dasonet_final_qc"
    / SEED1_QC.name,
)


# ============================================================
# 11. COPY FINAL SEED 2 COMPACT RECORD
# ============================================================

DST_SEED2_RUN = (
    STAGE_WORKSPACE
    / "dasonet_phase5b"
    / SEED2_RUN.name
)

copy_run_compact(
    SEED2_RUN,
    DST_SEED2_RUN,
    SEED2_TERMINAL,
)


copy_tree(
    SEED2_QC,
    STAGE_WORKSPACE
    / "dasonet_final_qc"
    / SEED2_QC.name,
)


# ============================================================
# 12. COPY DURATION / LEARNING EVIDENCE
# ============================================================

copy_file(
    DURATION_SUMMARY,
    STAGE_WORKSPACE
    / "dasonet_phase5c"
    / "duration_selection_val1000_v1"
    / DURATION_SUMMARY.name,
)


copy_file(
    LEARNING_CURVE_SUMMARY,
    STAGE_WORKSPACE
    / "dasonet_phase5c"
    / "baseline_curve1000updates_v1"
    / LEARNING_CURVE_SUMMARY.name,
)


# ============================================================
# 13. INNER STATE
# ============================================================

inner_state = {
    "handoff":
        "DASO-Net AFTER-SEED2",

    "date_tag":
        DATE_TAG,

    "scientific_state": {
        "seed1": {
            "rng_seed":
                SEED1,

            "status":
                "COMPLETE",

            "successful_optimizer_updates":
                2750,

            "terminal_sha256":
                SEED1_TERMINAL_SHA,
        },

        "seed2": {
            "rng_seed":
                SEED2,

            "status":
                "COMPLETE",

            "successful_optimizer_updates":
                2750,

            "terminal_sha256":
                SEED2_TERMINAL_SHA,

            "qc_status":
                "PASS",

            "qc_root":
                str(
                    SEED2_QC.relative_to(
                        WORKING
                    )
                ),
        },

        "next_seed": {
            "seed_index":
                3,

            "rng_seed":
                SEED3,

            "experiment_name":
                "dasonet_final_seed03_s252916394",

            "initialization":
                "fresh",

            "resume":
                "none",

            "target_successful_optimizer_updates":
                2750,
        },
    },

    "decoder_sha256":
        DECODER_SHA,

    "historical_original_recipe_zip_sha256":
        ORIGINAL_RECIPE_ZIP_SHA_PROVENANCE,

    "note":
        (
            "The historical recipe ZIP itself is intentionally "
            "not nested inside this handoff. The frozen extracted "
            "recipe directory is included instead, avoiding Kaggle "
            "archive-name collisions."
        ),
}


save_path = INNER_STATE

save_path.write_text(
    json.dumps(
        inner_state,
        indent=2,
    ),
    encoding="utf-8",
)


# ============================================================
# 14. README
# ============================================================

README.write_text(
    """
DASO-Net AFTER-SEED2 HANDOFF
============================

Purpose:
Restore the project state immediately after Final Seeds 1 and 2
for a fresh Final Seed-3 notebook.

Required Kaggle inputs in the new notebook:
1. abinavprabhuv/dasonet-project
2. a Kaggle dataset created from this AFTER-SEED2 handoff

Seed 3:
- seed index: 3
- RNG seed: 252916394
- fresh initialization
- resume: NONE
- target: exactly 2750 successful optimizer updates
- original PCC retained
- CRPA retained
- PCC-center disabled
- no external-data tuning
- no best-seed selection

This package intentionally contains NO nested recipe ZIP.
The frozen extracted recipe directory is included instead.
""".strip()
    + "\n",
    encoding="utf-8",
)


# ============================================================
# 15. RESTORE SCRIPT
# ============================================================

restore_source = r'''
from __future__ import annotations

import hashlib
import shutil
from pathlib import Path


PACKAGE_ROOT = Path(__file__).resolve().parent

SOURCE = (
    PACKAGE_ROOT
    / "workspace"
)

DEST = Path(
    "/kaggle/working"
)

EXPECTED_SEED1_SHA = (
    "e4c4b79ad4b8c6e18ba2b1e3d6cbe796c9cc301bd6fcc7c596fc290ad3a7e81d"
)

EXPECTED_SEED2_SHA = (
    "19cdb834a46240916b2c8023e8c3afe5738d9c7cd5d8f818c62ad1e289df1ad2"
)

SEED3_OUT = (
    DEST
    / "dasonet_phase5b"
    / "dasonet_final_seed03_s252916394"
)


def sha256_file(path: Path) -> str:
    h = hashlib.sha256()

    with path.open("rb") as f:
        for block in iter(
            lambda: f.read(4 * 1024 * 1024),
            b"",
        ):
            h.update(block)

    return h.hexdigest()


def merge_tree(
    src: Path,
    dst: Path,
) -> None:

    dst.mkdir(
        parents=True,
        exist_ok=True,
    )

    for item in src.iterdir():

        target = (
            dst
            / item.name
        )

        if item.is_dir():

            merge_tree(
                item,
                target,
            )

        else:

            target.parent.mkdir(
                parents=True,
                exist_ok=True,
            )

            shutil.copy2(
                item,
                target,
            )


if not SOURCE.exists():
    raise RuntimeError(
        f"Workspace source missing: {SOURCE}"
    )


if SEED3_OUT.exists():
    raise RuntimeError(
        "Seed-3 output already exists. "
        "Fresh initialization cannot be guaranteed."
    )


merge_tree(
    SOURCE,
    DEST,
)


seed1 = (
    DEST
    / "dasonet_phase5b"
    / "dasonet_final_seed01_s1782656963"
    / "snapshots"
    / "epoch_011_opt_002750.pt"
)

seed2 = (
    DEST
    / "dasonet_phase5b"
    / "dasonet_final_seed02_s901395089"
    / "snapshots"
    / "epoch_010_opt_002750.pt"
)


if sha256_file(seed1) != EXPECTED_SEED1_SHA:
    raise RuntimeError(
        "Restored Seed-1 terminal SHA mismatch."
    )

if sha256_file(seed2) != EXPECTED_SEED2_SHA:
    raise RuntimeError(
        "Restored Seed-2 terminal SHA mismatch."
    )


if SEED3_OUT.exists():
    raise RuntimeError(
        "Seed-3 output appeared unexpectedly."
    )


print(
    "=============================================="
)
print(
    "DASO-NET AFTER-SEED2 RESTORE: PASS"
)
print(
    "=============================================="
)
print(
    "Seed 1 terminal SHA: PASS"
)
print(
    "Seed 2 terminal SHA: PASS"
)
print(
    "Seed 3 output absent: PASS"
)
print(
    "Next Seed: 3"
)
print(
    "RNG seed: 252916394"
)
print(
    "Initialization: FRESH"
)
print(
    "Resume: NONE"
)
print(
    "Target successful optimizer updates: 2750"
)
'''

RESTORE_SCRIPT.write_text(
    restore_source.strip()
    + "\n",
    encoding="utf-8",
)


# ============================================================
# 16. BUILD FILE MANIFEST
# ============================================================

records = tree_manifest(
    STAGE_ROOT
)


manifest = {
    "package_name":
        PACKAGE_NAME,

    "file_count":
        len(records),

    "files":
        records,
}


INNER_MANIFEST.write_text(
    json.dumps(
        manifest,
        indent=2,
    ),
    encoding="utf-8",
)


# Rebuild records including manifest hash for audit reporting.
all_stage_files = [
    p
    for p in STAGE_ROOT.rglob("*")
    if p.is_file()
]


print()
print(
    "Staged files:",
    len(all_stage_files),
)


# ============================================================
# 17. CREATE ZIP
# ============================================================

with zipfile.ZipFile(
    ZIP_PATH,
    "w",
    compression=zipfile.ZIP_DEFLATED,
    compresslevel=6,
) as zf:

    for path in sorted(
        all_stage_files
    ):

        relative = (
            Path(PACKAGE_NAME)
            / path.relative_to(
                STAGE_ROOT
            )
        )

        # Checkpoints are already compressed-ish binary data.
        compression = (
            zipfile.ZIP_STORED
            if path.suffix.lower()
            in {
                ".pt",
                ".pth",
            }
            else zipfile.ZIP_DEFLATED
        )

        zf.write(
            path,
            arcname=str(
                relative
            ),
            compress_type=compression,
        )


# ============================================================
# 18. CRC AUDIT
# ============================================================

with zipfile.ZipFile(
    ZIP_PATH,
    "r",
) as zf:

    bad = zf.testzip()

    if bad is not None:

        raise RuntimeError(
            f"ZIP CRC failure: {bad}"
        )


# ============================================================
# 19. FULL EXTRACTED-BYTE AUDIT
# ============================================================

with tempfile.TemporaryDirectory() as tmp:

    tmp_root = Path(tmp)

    with zipfile.ZipFile(
        ZIP_PATH,
        "r",
    ) as zf:

        zf.extractall(
            tmp_root
        )

    extracted_root = (
        tmp_root
        / PACKAGE_NAME
    )

    extracted_manifest = json.loads(
        (
            extracted_root
            / "HANDOFF_MANIFEST.json"
        ).read_text(
            encoding="utf-8"
        )
    )

    for record in extracted_manifest[
        "files"
    ]:

        path = (
            extracted_root
            / record[
                "relative_path"
            ]
        )

        if not path.exists():

            raise RuntimeError(
                "ZIP staged-file missing:\n"
                f"{path}"
            )

        actual = sha256_file(
            path
        )

        if (
            actual
            != record[
                "sha256"
            ]
        ):

            raise RuntimeError(
                "ZIP staged-file SHA mismatch:\n"
                f"{path}"
            )


# ============================================================
# 20. OUTER STATE + SHA FILE
# ============================================================

zip_sha = sha256_file(
    ZIP_PATH
)


outer_state = {
    "package":
        ZIP_PATH.name,

    "package_sha256":
        zip_sha,

    "package_size_bytes":
        ZIP_PATH.stat().st_size,

    "crc":
        "PASS",

    "staged_file_sha_audit":
        "PASS",

    "seed1_terminal_sha256":
        SEED1_TERMINAL_SHA,

    "seed2_terminal_sha256":
        SEED2_TERMINAL_SHA,

    "seed2_qc_summary_sha256":
        sha256_file(
            SEED2_QC_SUMMARY
        ),

    "multiseed_plan_sha256":
        sha256_file(
            MULTISEED_PLAN
        ),

    "seed3_config_sha256":
        sha256_file(
            SEED3_CONFIG
        ),

    "decoder_sha256":
        DECODER_SHA,

    "next_seed":
        {
            "index":
                3,

            "rng_seed":
                SEED3,

            "fresh_initialization":
                True,

            "resume":
                "none",

            "target_successful_optimizer_updates":
                2750,
        },

    "required_new_notebook_inputs":
        [
            "abinavprabhuv/dasonet-project",
            PACKAGE_NAME,
        ],
}


STATE_PATH.write_text(
    json.dumps(
        outer_state,
        indent=2,
    ),
    encoding="utf-8",
)


SHA_PATH.write_text(
    f"{zip_sha}  {ZIP_PATH.name}\n",
    encoding="utf-8",
)


# ============================================================
# 21. FINAL REPORT
# ============================================================

print()
print(
    "============================================================"
)

print(
    "DASO-NET AFTER-SEED2 HANDOFF: PASS"
)

print(
    "============================================================"
)

print()
print(
    "ZIP:"
)
print(
    ZIP_PATH
)

print()
print(
    "Size:"
)
print(
    f"{ZIP_PATH.stat().st_size / (1024**2):.2f} MiB"
)

print()
print(
    "SHA256:"
)
print(
    zip_sha
)

print()
print(
    "SHA file:"
)
print(
    SHA_PATH
)

print()
print(
    "State:"
)
print(
    STATE_PATH
)

print()
print(
    "CRC: PASS"
)

print(
    "Staged SHA audit: PASS"
)

print()
print(
    "Seed 1 terminal: PASS"
)

print(
    "Seed 2 terminal: PASS"
)

print(
    "Seed 2 QC: PASS"
)

print(
    "Seed 3 frozen config: PASS"
)

print(
    "Seed 3 output absent: PASS"
)

print()
print(
    "NEXT FINAL SEED:"
)

print(
    "  index:                  3 / 3"
)

print(
    "  RNG seed:               252916394"
)

print(
    "  initialization:         FRESH"
)

print(
    "  resume:                 NONE"
)

print(
    "  target successful step: 2750"
)

print()
print(
    "DOWNLOAD THESE THREE FILES:"
)

print(
    f"1. {ZIP_PATH.name}"
)

print(
    f"2. {SHA_PATH.name}"
)

print(
    f"3. {STATE_PATH.name}"
)

print()
print(
    "DO NOT START SEED 3 IN THIS NOTEBOOK."
)

DASO-Net — AFTER-SEED2 HANDOFF BUILD
PASS  Validated code tree
PASS  Frozen multi-seed plan
PASS  Frozen recipe directory
PASS  Source training config
PASS  After-Seed1 restore audit
PASS  Seed-1 terminal
PASS  Seed-1 QC
PASS  Seed-2 terminal
PASS  Seed-2 final QC
PASS  Seed-2 QC summary
PASS  Seed-2 QC manifest
PASS  Duration-selection summary
PASS  Learning-curve summary
PASS  Validated decoder
PASS  Frozen Seed-3 config
PASS  Frozen multi-seed plan JSON

Seed-1 terminal SHA: PASS
Seed-2 terminal SHA: PASS
Decoder SHA: PASS
Seed-2 final QC: PASS
Seed-3 RNG 252916394: PASS
Seed-3 fresh-training budget 2750: PASS
Seed-3 output absent: PASS

Staged files: 85

DASO-NET AFTER-SEED2 HANDOFF: PASS

ZIP:
/kaggle/working/DASO-Net-AFTER-SEED2-HANDOFF-20260918.zip

Size:
41.45 MiB

SHA256:
3c5c3b9162052cca7ab9cc12c6ee2ffc6a738b06d77bde7d18fe193be6a6c5dd

SHA file:
/kaggle/working/DASO-Net-AFTER-SEED2-HANDOFF-20260918.sha256.txt

State:
/kaggle/working/DASO-Net-AFTER-SEED2-HANDOFF-20260918-STATE